# Prepare


In [ ]:
!pip -q install ortools pandas numpy tabulate

In [ ]:
import os
import math
import time
import random
from collections import defaultdict, Counter
from itertools import product

import numpy as np
import pandas as pd

from tabulate import tabulate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"

CNN_CSV   = os.path.join(BASE_DIR, "cnn_bench.csv")
VIT_CSV   = os.path.join(BASE_DIR, "vit_bench.csv")
BERT_CSV  = os.path.join(BASE_DIR, "bert_bench.csv")
GPT2_CSV  = os.path.join(BASE_DIR, "gpt2m_streaming_bench.csv")
LLAMA_CSV = os.path.join(BASE_DIR, "llama32_3b_streaming_bench.csv")

for p in [CNN_CSV, VIT_CSV, BERT_CSV, GPT2_CSV, LLAMA_CSV]:
    print(p, "->", os.path.exists(p))

In [ ]:
PROFILE_ORDER = ["7g", "4g", "3g", "2g", "1g"]

PROFILE_MEM_MB = {
    "1g": 5 * 1024,
    "2g": 10 * 1024,
    "3g": 20 * 1024,
    "4g": 20 * 1024,
    "7g": 40 * 1024,
}

In [ ]:
TEMPLATES = [
    ("7",               (1, 0, 0, 0, 0)),
    ("4+3",             (0, 1, 1, 0, 0)),
    ("4+2+1",           (0, 1, 0, 1, 1)),
    ("4+1+1+1",         (0, 1, 0, 0, 3)),
    ("3+3",             (0, 0, 2, 0, 0)),
    ("3+2+1",           (0, 0, 1, 1, 1)),
    ("3+1+1+1",         (0, 0, 1, 0, 3)),
    ("2+2+3",           (0, 0, 1, 2, 0)), # Config 8
    ("3+2+1+1",         (0, 0, 1, 1, 2)), # Config 9, 10
    ("3+1+1+1+1",       (0, 0, 1, 0, 4)), # Config 11
    ("2+2+2+1",         (0, 0, 0, 3, 1)), # Config 12
    ("2+2+1+1+1",       (0, 0, 0, 2, 3)), # Config 13, 14
    ("2+1+1+1+1+1",     (0, 0, 0, 1, 5)), # Config 15, 16, 17, 18
    ("1+1+1+1+1+1+1",   (0, 0, 0, 0, 7)), # Config 19
]

TEMPLATE_K = []
for name, vec in TEMPLATES:
    TEMPLATE_K.append({
        "7g": vec[0],
        "4g": vec[1],
        "3g": vec[2],
        "2g": vec[3],
        "1g": vec[4],
    })

print("Number of templates =", len(TEMPLATES))
for i, (name, _) in enumerate(TEMPLATES):
    print(f"{i:2d}: {name:18s} -> {TEMPLATE_K[i]}")

In [ ]:
WORKLOAD_SPECS = [
    {
        "name": "llama",
        "family": "llm",
        "csv": LLAMA_CSV,
        "model_match": "Llama-3.2-3B-Instruct",
        "batch_candidates": None,   # auto-discover from CSV
        "prompt_len": 1024,
        "output_tokens": 64,
        "arrival_rate": 3,       # req/s
        "slo_type": "TTFT/TPOT",
        "ttft_slo_ms": 100.0,
        "tpot_slo_ms": 25.0,
    },
    {
        "name": "gpt2",
        "family": "llm",
        "csv": GPT2_CSV,
        "model_match": "gpt2-medium",
        "batch_candidates": None,
        "prompt_len": 64,
        "output_tokens": 64,
        "arrival_rate": 20, #20
        "slo_type": "TTFT/TPOT",
        "ttft_slo_ms": 20.0,
        "tpot_slo_ms": 15.0,
    },
    {
        "name": "vgg16",
        "family": "cv",
        "csv": CNN_CSV,
        "model_match": "vgg16",
        "batch_candidates": None,
        "arrival_rate": 300, #300
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
    {
        "name": "resnet50",
        "family": "cv",
        "csv": CNN_CSV,
        "model_match": "resnet50",
        "batch_candidates": None,
        "arrival_rate": 300, #200
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
    {
        "name": "vit_base",
        "family": "cv",
        "csv": VIT_CSV,
        "model_match": "vit-base-patch16-224",
        "batch_candidates": None,
        "arrival_rate": 3000.0, #350
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
]

WORKLOAD_NAMES = [w["name"] for w in WORKLOAD_SPECS]
N_WORKLOADS = len(WORKLOAD_SPECS)

In [ ]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_mig_name(s: str):
    """
    Map strings like:
      'NVIDIA A100-PCIE-40GB MIG 1g.5gb' -> '1g'
      'MIG 3g.20gb' -> '3g'
    """
    if not isinstance(s, str):
        return None
    s = s.lower().replace(" ", "")
    if "1g." in s or "1g" in s:
        return "1g"
    if "2g." in s or "2g" in s:
        return "2g"
    if "3g." in s or "3g" in s:
        return "3g"
    if "4g." in s or "4g" in s:
        return "4g"
    if "7g." in s or "7g" in s:
        return "7g"
    return None


def decode_tps_to_tpot_ms(decode_tps: float):
    """
    decode_tps = tokens / second
    TPOT(ms/token) = 1000 / decode_tps
    """
    if decode_tps is None or np.isnan(decode_tps) or decode_tps <= 0:
        return np.nan
    return 1000.0 / decode_tps


def safe_req_rate_from_latency_ms(lat_ms: float, batch: int):
    """
    service rate (req/s) = 1000 / latency_ms
    """
    if lat_ms is None or np.isnan(lat_ms) or lat_ms <= 0:
        return np.nan
    return batch * 1000.0 / lat_ms

In [ ]:
def discover_batch_candidates_for_spec(spec, profile_order):
    df = pd.read_csv(spec["csv"]).copy()

    mig_col = pick_col(df, ["mig_name"])
    model_col = pick_col(df, ["model"])
    batch_col = pick_col(df, ["batch"])

    if mig_col is None or model_col is None or batch_col is None:
        raise ValueError(f"Missing essential columns in {spec['csv']}")

    df["profile"] = df[mig_col].apply(normalize_mig_name)
    df = df[df["profile"].isin(profile_order)].copy()
    df = df[df[model_col] == spec["model_match"]].copy()

    if spec["family"] == "llm":
        prompt_col = pick_col(df, ["prompt_len"])
        output_col = pick_col(df, ["output_tokens"])
        if prompt_col is None or output_col is None:
            raise ValueError(f"Missing LLM columns in {spec['csv']}")
        df = df[df[prompt_col] == spec["prompt_len"]].copy()
        df = df[df[output_col] == spec["output_tokens"]].copy()

    batches = sorted([int(b) for b in df[batch_col].dropna().unique().tolist() if float(b) > 0])

    if spec.get("batch_candidates") is not None:
        allow = set(int(x) for x in spec["batch_candidates"])
        batches = [b for b in batches if b in allow]

    return batches

In [ ]:
BATCHES_PER_WORKLOAD = {}
for spec in WORKLOAD_SPECS:
    BATCHES_PER_WORKLOAD[spec["name"]] = discover_batch_candidates_for_spec(spec, PROFILE_ORDER)

print("=== Batch candidates per workload ===")
for k, v in BATCHES_PER_WORKLOAD.items():
    print(f"{k:10s} -> {v}")

BATCH_LEVELS = sorted(set(b for lst in BATCHES_PER_WORKLOAD.values() for b in lst))
BATCH_TO_IDX = {b: idx for idx, b in enumerate(BATCH_LEVELS)}
IDX_TO_BATCH = {idx: b for b, idx in BATCH_TO_IDX.items()}

print("\nGlobal batch levels =", BATCH_LEVELS)
print("N_BATCH_LEVELS =", len(BATCH_LEVELS))

In [ ]:
def extract_rows_for_workload_batch(spec, batch_value, profile_order):
    """
    Return: dict profile -> stats for this (workload, batch)
    """
    df = pd.read_csv(spec["csv"]).copy()

    mig_col = pick_col(df, ["mig_name"])
    model_col = pick_col(df, ["model"])
    batch_col = pick_col(df, ["batch"])
    peak_mem_col = pick_col(df, ["peak_reserved_mb", "peak_alloc_mb"])
    time_mean_col = pick_col(df, ["time_ms_mean"])

    if mig_col is None or model_col is None or batch_col is None:
        raise ValueError(f"Missing essential columns in {spec['csv']}")

    df["profile"] = df[mig_col].apply(normalize_mig_name)
    df = df[df["profile"].isin(profile_order)].copy()
    df = df[df[model_col] == spec["model_match"]].copy()
    df = df[df[batch_col] == batch_value].copy()

    out = {}

    if spec["family"] == "cv":
        if len(df) == 0:
            return out

        g = df.groupby("profile", as_index=False)[time_mean_col].idxmin()
        chosen = df.loc[g[time_mean_col].values].copy()

        for _, row in chosen.iterrows():
            p = row["profile"]
            latency_ms = float(row[time_mean_col])
            peak_mb = float(row[peak_mem_col]) if peak_mem_col is not None else np.nan

            mem_ok = (peak_mb <= PROFILE_MEM_MB[p]) if not np.isnan(peak_mb) else False
            slo_ok = (latency_ms <= spec["e2e_slo_ms"])
            fit = bool(mem_ok and slo_ok)

            mu = safe_req_rate_from_latency_ms(latency_ms, batch_value) if fit else np.nan

            out[p] = {
                "batch": int(batch_value),
                "latency_ms": latency_ms,
                "peak_mem_mb": peak_mb,
                "ttft_ms": np.nan,
                "tpot_ms": np.nan,
                "service_time_ms": latency_ms,
                "mu_req_per_s": mu,
                "fit_mem": bool(mem_ok),
                "fit_slo": bool(slo_ok),
                "fit": fit,
            }

    elif spec["family"] == "llm":
        prompt_col = pick_col(df, ["prompt_len"])
        output_col = pick_col(df, ["output_tokens"])
        ttft_col = pick_col(df, ["ttft_ms"])
        decode_tps_col = pick_col(df, ["decode_tps"])

        if prompt_col is None or output_col is None or ttft_col is None or decode_tps_col is None:
            raise ValueError(f"Missing LLM columns in {spec['csv']}")

        df = df[df[prompt_col] == spec["prompt_len"]].copy()
        df = df[df[output_col] == spec["output_tokens"]].copy()

        if len(df) == 0:
            return out

        chosen = df.groupby("profile", as_index=False).first()

        for _, row in chosen.iterrows():
            p = row["profile"]
            ttft_ms = float(row[ttft_col])
            decode_tps = float(row[decode_tps_col])
            tpot_ms = decode_tps_to_tpot_ms(decode_tps)
            peak_mb = float(row[peak_mem_col]) if peak_mem_col is not None else np.nan

            mem_ok = (peak_mb <= PROFILE_MEM_MB[p]) if not np.isnan(peak_mb) else False
            slo_ok = (ttft_ms <= spec["ttft_slo_ms"]) and (tpot_ms <= spec["tpot_slo_ms"])
            fit = bool(mem_ok and slo_ok)

            # 这里沿用你之前的近似：service_time = TTFT + output_tokens * TPOT
            service_time_ms = ttft_ms + spec["output_tokens"] * tpot_ms if np.isfinite(tpot_ms) else np.nan
            mu = safe_req_rate_from_latency_ms(service_time_ms, batch_value) if fit else np.nan

            out[p] = {
                "batch": int(batch_value),
                "latency_ms": np.nan,
                "peak_mem_mb": peak_mb,
                "ttft_ms": ttft_ms,
                "tpot_ms": tpot_ms,
                "service_time_ms": service_time_ms,
                "mu_req_per_s": mu,
                "fit_mem": bool(mem_ok),
                "fit_slo": bool(slo_ok),
                "fit": fit,
            }

    else:
        raise ValueError(f"Unknown family: {spec['family']}")

    return out

In [ ]:
def build_workload_batch_profile_tensors(workload_specs, profile_order, batch_levels, batches_per_workload):
    W = len(workload_specs)
    B = len(batch_levels)
    P = len(profile_order)

    mu = np.full((W, B, P), np.nan, dtype=float)
    service_time_ms = np.full((W, B, P), np.nan, dtype=float)
    latency_ms = np.full((W, B, P), np.nan, dtype=float)
    ttft_ms = np.full((W, B, P), np.nan, dtype=float)
    tpot_ms = np.full((W, B, P), np.nan, dtype=float)
    peak_mem_mb = np.full((W, B, P), np.nan, dtype=float)

    fit_mem = np.zeros((W, B, P), dtype=bool)
    fit_slo = np.zeros((W, B, P), dtype=bool)
    fit = np.zeros((W, B, P), dtype=bool)
    batch_allowed = np.zeros((W, B), dtype=bool)

    raw = {}

    for i, spec in enumerate(workload_specs):
        wname = spec["name"]
        raw[wname] = {}

        for batch_value in batches_per_workload[wname]:
            bidx = BATCH_TO_IDX[batch_value]
            batch_allowed[i, bidx] = True

            row_dict = extract_rows_for_workload_batch(spec, batch_value, profile_order)
            raw[wname][batch_value] = row_dict

            for j, p in enumerate(profile_order):
                if p not in row_dict:
                    continue

                d = row_dict[p]
                mu[i, bidx, j] = d["mu_req_per_s"]
                service_time_ms[i, bidx, j] = d["service_time_ms"]
                latency_ms[i, bidx, j] = d["latency_ms"]
                ttft_ms[i, bidx, j] = d["ttft_ms"]
                tpot_ms[i, bidx, j] = d["tpot_ms"]
                peak_mem_mb[i, bidx, j] = d["peak_mem_mb"]
                fit_mem[i, bidx, j] = d["fit_mem"]
                fit_slo[i, bidx, j] = d["fit_slo"]
                fit[i, bidx, j] = d["fit"]

    return {
        "mu": mu,
        "service_time_ms": service_time_ms,
        "latency_ms": latency_ms,
        "ttft_ms": ttft_ms,
        "tpot_ms": tpot_ms,
        "peak_mem_mb": peak_mem_mb,
        "fit_mem": fit_mem,
        "fit_slo": fit_slo,
        "fit": fit,
        "batch_allowed": batch_allowed,
        "raw": raw,
    }

In [ ]:
mat3 = build_workload_batch_profile_tensors(
    workload_specs=WORKLOAD_SPECS,
    profile_order=PROFILE_ORDER,
    batch_levels=BATCH_LEVELS,
    batches_per_workload=BATCHES_PER_WORKLOAD,
)

mu_3d = mat3["mu"]
service_time_ms_3d = mat3["service_time_ms"]
latency_ms_3d = mat3["latency_ms"]
ttft_ms_3d = mat3["ttft_ms"]
tpot_ms_3d = mat3["tpot_ms"]
peak_mem_mb_3d = mat3["peak_mem_mb"]
fit_mem_3d = mat3["fit_mem"]
fit_slo_3d = mat3["fit_slo"]
fit_3d = mat3["fit"]
batch_allowed = mat3["batch_allowed"]

arrival_rate = np.array([w["arrival_rate"] for w in WORKLOAD_SPECS], dtype=float)

print("mu_3d shape =", mu_3d.shape)              # (W, B, P)
print("fit_3d shape =", fit_3d.shape)            # (W, B, P)
print("batch_allowed shape =", batch_allowed.shape)

In [ ]:
def build_option_table(workload_specs, profile_order, batch_levels, mu_3d, fit_3d,
                       fit_mem_3d, fit_slo_3d, peak_mem_mb_3d, service_time_ms_3d,
                       latency_ms_3d, ttft_ms_3d, tpot_ms_3d):
    rows = []

    for i, spec in enumerate(workload_specs):
        for bidx, b in enumerate(batch_levels):
            for pidx, p in enumerate(profile_order):
                rows.append({
                    "w_idx": i,
                    "workload": spec["name"],
                    "family": spec["family"],
                    "batch_idx": bidx,
                    "batch": int(b),
                    "p_idx": pidx,
                    "profile": p,
                    "mu": mu_3d[i, bidx, pidx],
                    "service_time_ms": service_time_ms_3d[i, bidx, pidx],
                    "latency_ms": latency_ms_3d[i, bidx, pidx],
                    "ttft_ms": ttft_ms_3d[i, bidx, pidx],
                    "tpot_ms": tpot_ms_3d[i, bidx, pidx],
                    "peak_mem_mb": peak_mem_mb_3d[i, bidx, pidx],
                    "fit_mem": bool(fit_mem_3d[i, bidx, pidx]),
                    "fit_slo": bool(fit_slo_3d[i, bidx, pidx]),
                    "fit": bool(fit_3d[i, bidx, pidx]),
                })

    option_df = pd.DataFrame(rows)

    feasible_option_df = option_df[option_df["fit"]].copy().reset_index(drop=True)
    feasible_option_df["opt_idx"] = np.arange(len(feasible_option_df))

    return option_df, feasible_option_df

In [ ]:
option_df, feasible_option_df = build_option_table(
    workload_specs=WORKLOAD_SPECS,
    profile_order=PROFILE_ORDER,
    batch_levels=BATCH_LEVELS,
    mu_3d=mu_3d,
    fit_3d=fit_3d,
    fit_mem_3d=fit_mem_3d,
    fit_slo_3d=fit_slo_3d,
    peak_mem_mb_3d=peak_mem_mb_3d,
    service_time_ms_3d=service_time_ms_3d,
    latency_ms_3d=latency_ms_3d,
    ttft_ms_3d=ttft_ms_3d,
    tpot_ms_3d=tpot_ms_3d,
)

print("All options =", len(option_df))
print("Feasible options =", len(feasible_option_df))

display(feasible_option_df)

In [ ]:
FEASIBLE_OPTIONS_BY_WORKLOAD = {
    i: feasible_option_df[feasible_option_df["w_idx"] == i]["opt_idx"].tolist()
    for i in range(N_WORKLOADS)
}

FEASIBLE_OPTIONS_BY_PROFILE = {
    j: feasible_option_df[feasible_option_df["p_idx"] == j]["opt_idx"].tolist()
    for j in range(len(PROFILE_ORDER))
}

print("=== #feasible options by workload ===")
for i, w in enumerate(WORKLOAD_NAMES):
    print(f"{w:10s} -> {len(FEASIBLE_OPTIONS_BY_WORKLOAD[i])}")

In [ ]:
def show_feasible_options_per_workload(feasible_option_df):
    rows = []
    for w, g in feasible_option_df.groupby("workload"):
        items = []
        for _, r in g.sort_values(["batch", "profile"]).iterrows():
            items.append(f"(b={int(r['batch'])}, p={r['profile']}, mu={r['mu']:.4f})")
        rows.append([w, len(g), "; ".join(items[:20]) + (" ..." if len(items) > 20 else "")])

    print(tabulate(
        rows,
        headers=["Workload", "#Feasible (batch,profile) options", "Options"],
        tablefmt="github"
    ))

show_feasible_options_per_workload(feasible_option_df)

In [ ]:
def check_global_feasibility_batch(workload_specs, feasible_option_df):
    ok = True
    rows = []

    for i, spec in enumerate(workload_specs):
        g = feasible_option_df[feasible_option_df["w_idx"] == i].copy()
        pairs = list(zip(g["batch"].tolist(), g["profile"].tolist()))
        rows.append([
            spec["name"],
            len(pairs),
            pairs[:15] if len(pairs) <= 15 else pairs[:15] + ["..."]
        ])
        if len(pairs) == 0:
            ok = False

    print(tabulate(
        rows,
        headers=["Workload", "#Feasible (batch,profile)", "Feasible pairs"],
        tablefmt="github"
    ))

    if ok:
        print("\n[OK] Every workload has at least one feasible (batch, profile) option.")
    else:
        print("\n[WARN] Some workload has no feasible (batch, profile) option. MILP will be infeasible.")

check_global_feasibility_batch(WORKLOAD_SPECS, feasible_option_df)

In [ ]:
def print_experiment_header_batch():
    print("=" * 90)
    print("Simulation: Min #GPU under arrival-rate / SLO / MIG constraints (batch-selectable)")
    print("=" * 90)
    print(f"Workloads      : {WORKLOAD_NAMES}")
    print(f"Profiles       : {PROFILE_ORDER}")
    print(f"Batch levels   : {BATCH_LEVELS}")
    print(f"#Templates     : {len(TEMPLATES)}")
    print()

    rows = []
    for w in WORKLOAD_SPECS:
        if w["family"] == "llm":
            slo_str = f"TTFT<={w['ttft_slo_ms']}ms, TPOT<={w['tpot_slo_ms']}ms"
            extra = f"prompt={w['prompt_len']}, out={w['output_tokens']}"
        else:
            slo_str = f"E2E<={w['e2e_slo_ms']}ms"
            extra = "-"

        rows.append([
            w["name"],
            w["family"],
            BATCHES_PER_WORKLOAD[w["name"]],
            w["arrival_rate"],
            slo_str,
            extra
        ])

    print(tabulate(
        rows,
        headers=["Workload", "Family", "Batch candidates", "Arrival(req/s)", "SLO", "Extra"],
        tablefmt="github",
        floatfmt=".4f"
    ))

print_experiment_header_batch()

In [ ]:
def format_instance_list(instances):
    items = []
    for inst in instances:
        items.append(f"({inst['batch']}, {inst['profile']}, {inst['count']})")
    return "; ".join(items)

def print_solution_summary(
    method_name,
    elapsed,
    feasible,
    objective,
    K_total,
    templates_used,
    alloc
):
    print("=" * 90)
    print(f"Method        : {method_name}")
    print(f"Elapsed (s)   : {elapsed:.4f}")
    print(f"Feasible      : {feasible}")
    #print(f"GPU count     : {sum(K_total.values())}")
    print(f"GPU count     : {len(templates_used)}")
    print(f"Objective     : {objective:.4f}" if objective is not None else "Objective     : -")
    print(f"Templates     : {templates_used}")
    print(f"K_total       : {K_total}")
    print("=" * 90)


    rows = []
    for w in alloc:
        ratio = w["provided"] / w["arrival"] if w["arrival"] > 0 else 0.0

        rows.append([
            w["workload"],
            f"{w['arrival']:.4f}",
            f"{w['provided']:.4f}",
            f"{ratio:.3f}",
            f"{w['slack']:.4f}",
            format_instance_list(w["instances"])
        ])

    print("\nPer-workload allocation / rate summary:")
    print(tabulate(
        rows,
        headers=[
            "Workload",
            "Arrival",
            "Provided",
            "Provided/Arrival",
            "Slack",
            "Placement (batch,profile,count)"
        ],
        tablefmt="github"
    ))

In [ ]:
def build_allocation_from_x(
    feasible_option_df,
    x_sol,
    arrival_rate
):
    """
    x_sol: dict {opt_idx: value}
    """

    alloc = []

    for i in range(len(arrival_rate)):
        g = feasible_option_df[feasible_option_df["w_idx"] == i]

        instances = []
        provided = 0.0

        for _, row in g.iterrows():
            o = row["opt_idx"]
            x_val = x_sol.get(o, 0)

            if x_val <= 0:
                continue

            mu = row["mu"]
            provided += x_val * mu

            instances.append({
                "batch": int(row["batch"]),
                "profile": row["profile"],
                "count": int(x_val),
                "mu": mu
            })

        arrival = arrival_rate[i]
        slack = provided - arrival

        alloc.append({
            "workload": g["workload"].iloc[0],
            "arrival": arrival,
            "provided": provided,
            "slack": slack,
            "instances": instances
        })

    return alloc

In [ ]:
def compute_slo_violation_rate(alloc):
    rows = []

    for w in alloc:
        if w["arrival"] <= 0:
            viol = 0.0
        else:
            viol = max(0.0, 1.0 - w["provided"] / w["arrival"])

        rows.append([
            w["workload"],
            f"{viol:.4f}"
        ])

    print("\nSLO Violation Rate per workload:")
    print(tabulate(
        rows,
        headers=["Workload", "Violation Rate"],
        tablefmt="github"
    ))

# MILP

In [ ]:
!pip install gurobipy

In [ ]:

import time
import math
import numpy as np
from tabulate import tabulate
import gurobipy as gp
from gurobipy import GRB


def milp_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def milp_build_K_total(y_sol, template_K, profile_order):
    out = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            out[p] += int(cnt) * int(template_K[t_idx][p])
    return out


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0
    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal
        rows.append([o, row["workload"], int(row["batch"]), row["profile"], xv, f"{mu_o:.6f}", f"{subtotal:.6f}"])

    print("\nChosen instances:")
    print(tabulate(rows, headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"], tablefmt="github"))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


def prune_dominated_options(feasible_option_df):
    """
    Workload-wise Pareto pruning.
    Drop option B if there exists option A for the same workload with:
      mu_A >= mu_B and profile_size(A) <= profile_size(B),
    with at least one strict improvement.
    Keep opt_idx unchanged for compatibility with downstream extraction/warm-start.
    """
    def _profile_size_from_name(p):
        s = str(p)
        if s == "void":
            return 0
        return int(s.replace("g", ""))

    df = feasible_option_df.copy()
    keep_mask = np.ones(len(df), dtype=bool)

    for w_idx, grp in df.groupby("w_idx", sort=False):
        rows = grp[["opt_idx", "mu", "profile"]].copy()
        rows["size"] = rows["profile"].map(_profile_size_from_name).astype(int)

        idxs = rows.index.tolist()
        for i in idxs:
            if not keep_mask[i]:
                continue
            mu_i = float(rows.loc[i, "mu"])
            sz_i = int(rows.loc[i, "size"])

            dominated = False
            for j in idxs:
                if i == j:
                    continue
                mu_j = float(rows.loc[j, "mu"])
                sz_j = int(rows.loc[j, "size"])

                if (mu_j >= mu_i and sz_j <= sz_i) and (mu_j > mu_i or sz_j < sz_i):
                    dominated = True
                    break

            if dominated:
                keep_mask[i] = False

    pruned = df.loc[keep_mask].copy().reset_index(drop=True)
    return pruned


def compute_elastic_up_by_opt(base_option_df):
    """
    For each option r=(workload, profile, batch, mu), compute how much throughput
    can still be gained by increasing batch size while keeping the SAME profile.

    delta_mu_up[r] = max_{b' > b, same workload, same profile}(mu_{b'} - mu_b, 0)
    """
    base_df = base_option_df.copy()
    delta_map = {}

    # group by workload/profile on the FULL option table (before pruning)
    for (w_idx, profile), grp in base_df.groupby(["w_idx", "profile"], sort=False):
        grp2 = grp[["opt_idx", "batch", "mu"]].copy()
        grp2["batch"] = grp2["batch"].astype(int)
        grp2["mu"] = grp2["mu"].astype(float)
        grp2 = grp2.sort_values(["batch", "mu"]).reset_index(drop=True)

        batches = grp2["batch"].tolist()
        mus = grp2["mu"].tolist()
        opt_idxs = grp2["opt_idx"].tolist()

        # suffix max of future mu for strictly larger batch
        suffix_best = [0.0] * len(grp2)
        best = -float("inf")
        for i in range(len(grp2) - 1, -1, -1):
            suffix_best[i] = best
            best = max(best, mus[i])

        for i, opt_idx in enumerate(opt_idxs):
            cur_mu = mus[i]
            future_best_mu = suffix_best[i]
            if future_best_mu == -float("inf"):
                delta = 0.0
            else:
                delta = max(0.0, future_best_mu - cur_mu)
            delta_map[int(opt_idx)] = float(delta)

    # any missing key gets 0
    for opt_idx in base_df["opt_idx"].tolist():
        delta_map.setdefault(int(opt_idx), 0.0)

    return delta_map


def solve_milp_gurobi_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    verbose=False,
    apply_option_pruning=True,
    warm_start_res=None,
):
    start = time.time()

    base_df = feasible_option_df.copy()
    elastic_up_map = compute_elastic_up_by_opt(base_df)

    if apply_option_pruning:
        df = prune_dominated_options(base_df).reset_index(drop=True)
    else:
        df = base_df.reset_index(drop=True)

    # keep full-table opt_idx and attach elastic-up metadata
    df["elastic_up"] = df["opt_idx"].map(lambda o: float(elastic_up_map.get(int(o), 0.0)))

    n_opts = len(df)
    nT = len(templates)

    opt_rows = list(range(n_opts))
    t_ids = list(range(nT))

    options_by_workload = {i: [] for i in range(n_workloads)}
    options_by_profile = {p: [] for p in profile_order}

    for r in opt_rows:
        w_idx = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        options_by_workload[w_idx].append(r)
        options_by_profile[p].append(r)

    for i in range(n_workloads):
        if len(options_by_workload[i]) == 0:
            return {
                "method": "MILP-Gurobi(batch)",
                "feasible": False,
                "status": f"NO_OPTION_FOR_WORKLOAD_{i}",
                "elapsed": time.time() - start,
                "effective_option_df": df,
            }

    model = gp.Model("milp_batch_elastic")

    if not verbose:
        model.Params.OutputFlag = 0
    if time_limit_s is not None:
        model.Params.TimeLimit = float(time_limit_s)
    if mip_gap is not None:
        model.Params.MIPGap = float(mip_gap)
    if threads is not None:
        model.Params.Threads = int(threads)

    y = model.addVars(t_ids, vtype=GRB.INTEGER, lb=0, name="y")

    total_gpu = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_gpu")
    model.addConstr(total_gpu == gp.quicksum(y[t] for t in t_ids), name="def_total_gpu")

    cap = {}
    for p in profile_order:
        cap[p] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"cap_{p}")
        model.addConstr(
            cap[p] == gp.quicksum(int(template_K[t][p]) * y[t] for t in t_ids),
            name=f"def_cap_{p}"
        )

    ub_gpu_loose = 0
    for i in range(n_workloads):
        g = df[df["w_idx"] == i]
        best_mu = float(g["mu"].max())
        ub_gpu_loose += int(math.ceil(arrival_rate[i] / best_mu)) if best_mu > 0 else 0
    ub_gpu_loose = max(1, ub_gpu_loose)

    x = {}
    for r in opt_rows:
        w_idx = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        mu_o = float(df.loc[r, "mu"])

        ub_by_demand = int(math.ceil(arrival_rate[w_idx] / mu_o)) + 2 if mu_o > 0 else 0
        max_profile_per_gpu = max(int(template_K[t][p]) for t in t_ids)
        ub_by_gpu = ub_gpu_loose * max_profile_per_gpu
        ub = max(1, min(ub_by_demand, ub_by_gpu))

        x[r] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=ub, name=f"x_{r}")

    provided = {}
    slack = {}
    for i in range(n_workloads):
        provided[i] = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name=f"provided_{i}")
        slack[i] = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name=f"slack_{i}")

        expr = gp.quicksum(float(df.loc[r, "mu"]) * x[r] for r in options_by_workload[i])
        model.addConstr(provided[i] == expr, name=f"def_provided_{i}")
        model.addConstr(provided[i] >= float(arrival_rate[i]), name=f"demand_{i}")
        model.addConstr(slack[i] == provided[i] - float(arrival_rate[i]), name=f"def_slack_{i}")

    for p in profile_order:
        model.addConstr(
            gp.quicksum(x[r] for r in options_by_profile[p]) <= cap[p],
            name=f"profile_cap_{p}"
        )

    # diagnostic-only static slack (not the 2nd objective anymore)
    total_slack = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="total_slack")
    model.addConstr(total_slack == gp.quicksum(slack[i] for i in range(n_workloads)), name="def_total_slack")

    # NEW: elastic slack = future throughput gain reachable by increasing batch
    elastic_up = {}
    for r in opt_rows:
        elastic_up[r] = float(df.loc[r, "elastic_up"])

    total_elastic_slack = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="total_elastic_slack")
    model.addConstr(
        total_elastic_slack == gp.quicksum(elastic_up[r] * x[r] for r in opt_rows),
        name="def_total_elastic_slack",
    )

    total_instances = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_instances")
    model.addConstr(total_instances == gp.quicksum(x[r] for r in opt_rows), name="def_total_instances")

    remaining = {}
    for p in profile_order:
        remaining[p] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"remaining_{p}")
        model.addConstr(remaining[p] == cap[p] - gp.quicksum(x[r] for r in options_by_profile[p]), name=f"def_remaining_{p}")

    total_remaining_slots = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_remaining_slots")
    model.addConstr(
        total_remaining_slots == gp.quicksum(int(str(p).replace("g","")) * remaining[p] for p in profile_order),
        name="def_total_remaining_slots",
    )

    model.setObjectiveN(total_gpu, index=0, priority=3, weight=1.0, name="obj_gpu")
    model.setObjectiveN(total_elastic_slack, index=1, priority=2, weight=-1.0, name="obj_elastic_slack")
    model.setObjectiveN(total_remaining_slots, index=2, priority=1, weight=-1.0, name="obj_remaining")

    # ---- warm start from previous MILP result ----
    if warm_start_res is not None:
        prev_x = dict(warm_start_res.get("x_sol", {}) or {})
        prev_y = dict(warm_start_res.get("y_sol", {}) or {})
        global_to_local = {int(df.loc[r, "opt_idx"]): r for r in opt_rows}

        for t in t_ids:
            if t in prev_y:
                y[t].Start = float(prev_y[t])

        for global_opt_idx, val in prev_x.items():
            if global_opt_idx in global_to_local:
                r = global_to_local[global_opt_idx]
                x[r].Start = float(val)

    model.optimize()

    elapsed = time.time() - start
    status_str = {
        GRB.OPTIMAL: "OPTIMAL",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.SUBOPTIMAL: "SUBOPTIMAL",
        GRB.INFEASIBLE: "INFEASIBLE",
    }.get(model.Status, str(model.Status))

    if model.SolCount == 0:
        return {
            "method": "MILP-Gurobi(batch)",
            "feasible": False,
            "status": status_str,
            "elapsed": elapsed,
            "effective_option_df": df,
        }

    y_sol = {}
    for t in t_ids:
        v = int(round(y[t].X))
        if v > 0:
            y_sol[t] = v

    x_sol = {}
    for r in opt_rows:
        v = int(round(x[r].X))
        if v > 0:
            global_opt_idx = int(df.loc[r, "opt_idx"])
            x_sol[global_opt_idx] = v

    K_total = milp_build_K_total(y_sol, template_K, profile_order)
    chosen_templates = milp_expand_template_list(y_sol, templates)

    alloc = build_allocation_from_x(
        feasible_option_df=base_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)
    used_profile_types = sum(1 for p in profile_order if K_total[p] > 0)

    return {
        "method": "MILP-Gurobi(batch)",
        "feasible": True,
        "status": status_str,
        "elapsed": elapsed,
        "gpu_count": int(round(total_gpu.X)),
        "objective": int(round(total_gpu.X)),
        "chosen_templates": chosen_templates,
        "K_total": K_total,
        "x_sol": x_sol,
        "y_sol": y_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": float(total_slack.X),                       # diagnostic
        "total_elastic_slack": float(total_elastic_slack.X),     # objective-2 value
        "total_remaining_slots": int(round(total_remaining_slots.X)),
        "total_instances": int(round(total_instances.X)),
        "used_profile_types": used_profile_types,
        "effective_option_df": df,
    }


def print_milp_gurobi_batch_result(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        if "effective_option_df" in res and res["effective_option_df"] is not None:
            print(f"Effective opts : {len(res['effective_option_df'])}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nSolver status        : {res['status']}")
    print(f"Total static slack   : {res['total_slack']:.6f}")
    print(f"Total elastic slack  : {res.get('total_elastic_slack', 0.0):.6f}")
    print(f"Total instances      : {res['total_instances']}")
    print(f"Total remaining      : {res['total_remaining_slots']}")
    print(f"Used profile types   : {res['used_profile_types']}")
    if "effective_option_df" in res and res["effective_option_df"] is not None:
        print(f"Effective options    : {len(res['effective_option_df'])}")

    compute_slo_violation_rate(res["alloc"])


def print_milp_template_counts_unified(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])

    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))


# MILP result

In [ ]:
milp_res = solve_milp_gurobi_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    verbose=False,
)

print_milp_gurobi_batch_result(milp_res)

if milp_res["feasible"]:
    print_milp_template_counts_unified(milp_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, milp_res["x_sol"])



#Transition

## Transition 重构说明
下面的代码从 `#Transition` 之后开始完整替换原来的大代码块。

设计保持原语义：
- MILP 继续使用 **14 个 abstract templates**。
- 目标状态搜索继续使用 **19 个 legal physical configurations**。
- 保留 `void`、interval、MILP multiset 校验、输出字段等基础设施。

新增点：
- 将原来混在一个代码块里的基础设施、abstract 层、physical 层、beam、greedy 分开。
- 统一入口 `realize_target_state_search_from_milp(...)`，通过 `set_target_search_method(...)` 切换算法。


### Code Block 1: Imports / notebook-aligned constants / state classes


In [ ]:
# ============================================================
# [Transition Cell 1] Imports / notebook-aligned constants / state classes
# Role:
#   - import dependencies
#   - build template/profile lookup tables from prepare-stage variables
#   - define state classes used only after #transition
# ============================================================

from dataclasses import dataclass, field
from collections import defaultdict, Counter
from typing import Dict, Any, List, Optional, Tuple
import time
import math
import copy

# ---------- notebook-aligned constants ----------
if "PROFILE_ORDER" not in globals():
    PROFILE_ORDER = ["7g", "4g", "3g", "2g", "1g"]

PROFILE_SIZE = {
    "7g": 7,
    "4g": 4,
    "3g": 3,
    "2g": 2,
    "1g": 1,
    "void": 0,
}
SIZE_TO_PROFILE = {7: "7g", 4: "4g", 3: "3g", 2: "2g", 1: "1g"}

def _template_name_list_from_globals():
    return [name for name, _ in TEMPLATES]

def _template_capacity_dict(template_name: str) -> Dict[str, int]:
    for name, vec in TEMPLATES:
        if name == template_name:
            return {"7g": int(vec[0]), "4g": int(vec[1]), "3g": int(vec[2]), "2g": int(vec[3]), "1g": int(vec[4])}
    raise KeyError(f"Unknown template: {template_name}")

TEMPLATE_NAME_TO_K = {name: _template_capacity_dict(name) for name, _ in TEMPLATES}

def _template_to_parts(template_name: str) -> Tuple[int, ...]:
    return tuple(int(x) for x in template_name.split("+"))

def _parts_to_profiles(parts: Tuple[int, ...]) -> Tuple[str, ...]:
    return tuple(SIZE_TO_PROFILE[int(x)] for x in parts)

def _physical_profiles_to_string(profiles: Tuple[str, ...]) -> str:
    return "+".join(str(PROFILE_SIZE[p]) for p in profiles)

def _physical_profiles_to_intervals(profiles: Tuple[str, ...]) -> List[Tuple[int, int, str]]:
    out = []
    cur = 0
    for p in profiles:
        sz = PROFILE_SIZE[p]
        out.append((cur, cur + sz, p))
        cur += sz
    if cur < 7:
        out.append((cur, 7, "void"))
    return out

# 14 abstract templates -> 19 legal physical configurations
ABSTRACT_TO_PHYSICAL = {
    "7": [("7g",)],
    "4+3": [("4g", "3g")],
    "4+2+1": [("4g", "2g", "1g")],
    "4+1+1+1": [("4g", "1g", "1g", "1g")],
    "3+3": [("3g", "3g")],
    "3+2+1": [("3g", "2g", "1g")],
    "3+1+1+1": [("3g", "1g", "1g", "1g")],
    "2+2+3": [("2g", "2g", "3g")],
    "3+2+1+1": [
        ("2g", "1g", "1g", "3g"),
        ("1g", "1g", "2g", "3g"),
    ],
    "3+1+1+1+1": [("1g", "1g", "1g", "1g", "3g")],
    "2+2+2+1": [("2g", "2g", "2g", "1g")],
    "2+2+1+1+1": [
        ("2g", "1g", "1g", "2g", "1g"),
        ("1g", "1g", "2g", "2g", "1g"),
    ],
    "2+1+1+1+1+1": [
        ("2g", "1g", "1g", "1g", "1g", "1g"),
        ("1g", "1g", "2g", "1g", "1g", "1g"),
        ("1g", "1g", "1g", "1g", "2g", "1g"),
        ("1g", "1g", "1g", "1g", "1g", "2g"),
    ],
    "1+1+1+1+1+1+1": [("1g", "1g", "1g", "1g", "1g", "1g", "1g")],
}

def _all_unique_physical_realizations(template_name: str) -> List[Tuple[str, List[Tuple[int, int, str]]]]:
    if template_name not in ABSTRACT_TO_PHYSICAL:
        raise KeyError(f"Unknown abstract template: {template_name}")
    out = []
    for profiles in ABSTRACT_TO_PHYSICAL[template_name]:
        out.append((_physical_profiles_to_string(profiles), _physical_profiles_to_intervals(profiles)))
    return out

@dataclass
class MigInstance:
    start: int
    end: int
    profile: str
    workload: Optional[str] = None
    batch: Optional[int] = None
    mu: float = 0.0
    preserved: bool = False

@dataclass
class GPUState:
    gpu_id: int
    source: str = "real"
    instances: List[Any] = field(default_factory=list)

    def sort_instances(self):
        self.instances = sorted(self.instances, key=lambda x: (x.start, x.end, x.profile))

    def template_str(self):
        self.sort_instances()
        return "+".join(str(PROFILE_SIZE[inst.profile]) for inst in self.instances if inst.profile != "void")

@dataclass
class ClusterState:
    gpus: List[Any] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def real_gpus(self):
        return [g for g in self.gpus if getattr(g, "source", "real") == "real"]


### Code Block 2: Validation / pretty-print / verification


In [ ]:

# ============================================================
# [Transition Cell 2] Validation / pretty-print / verification
# Role:
#   - validate target state structure
#   - pretty-print searched target state
#   - verify searched target against MILP multiset
# Note:
#   verification is relaxed to allow upward placement
#   (e.g., a MILP 3g demand may be realized on a 4g slot)
# ============================================================

def assert_valid_cluster_state(state: "ClusterState"):
    for g in state.real_gpus():
        g.sort_instances()
        cur = 0
        for inst in g.instances:
            if inst.start != cur:
                raise ValueError(f"GPU {g.gpu_id}: expected start={cur}, got {inst.start}")
            if inst.end <= inst.start:
                raise ValueError(f"GPU {g.gpu_id}: bad interval ({inst.start},{inst.end})")
            if inst.profile != "void":
                if PROFILE_SIZE[inst.profile] != (inst.end - inst.start):
                    raise ValueError(f"GPU {g.gpu_id}: profile-size mismatch on {inst.profile}")
            cur = inst.end
        if cur != 7:
            raise ValueError(f"GPU {g.gpu_id}: total covered slices={cur}, expected 7")

def summarize_state(state: "ClusterState", title="[STATE]"):
    print("=" * 90)
    print(title)
    print("=" * 90)
    for g in state.real_gpus():
        row = []
        for inst in sorted(g.instances, key=lambda x: (x.start, x.end)):
            if inst.workload is None:
                row.append(f"[{inst.start},{inst.end}) {inst.profile}: free")
            else:
                row.append(
                    f"[{inst.start},{inst.end}) {inst.profile}: "
                    f"{inst.workload} / B{inst.batch} / mu={inst.mu:.4f}"
                )
        print(f"GPU {g.gpu_id}: " + " | ".join(row))

def print_target_search_result(target: "ClusterState", title="[CHECK] TARGET STATE"):
    summarize_state(target, title=title)
    metrics = target.metadata.get("search_metrics", {})
    if metrics:
        print("\nSearch metrics:")
        for k in [
            "elapsed_time_sec",
            "ordered_abstract_templates",
            "ordered_physical_templates",
            "layout_preserve_score",
            "exact_preserve",
            "same_gpu_preserve",
            "spread",
            "collocate_pairs",
            "mixed_gpu_count",
            "template_match_count",
            "score_tuple",
        ]:
            if k in metrics:
                print(f"  {k}: {metrics[k]}")

def _collect_instance_multiset_from_target(target: "ClusterState") -> Counter:
    cnt = Counter()
    for g in target.real_gpus():
        for inst in g.instances:
            if inst.workload is None:
                continue
            cnt[(inst.workload, inst.profile, int(inst.batch))] += 1
    return cnt

def _collect_instance_multiset_from_milp(milp_res: Dict[str, Any]) -> Counter:
    cnt = Counter()
    for d in extract_instance_demands_from_milp(milp_res):
        cnt[(d["workload"], d["profile"], int(d["batch"]))] += int(d["count"])
    return cnt

def _group_multiset_by_workload_batch(counter_obj: Counter):
    grouped = defaultdict(lambda: Counter())
    for (w, p, b), c in counter_obj.items():
        grouped[(w, int(b))][PROFILE_SIZE[p]] += int(c)
    return grouped

def _relaxed_cover_ok_for_one_group(target_size_cnt: Counter, demand_size_cnt: Counter) -> bool:
    """
    Relaxed verification only for the rewrite rule:
      3g demand may be realized on a 4g slot.

    All other profile sizes must match exactly.
    """
    available = Counter(target_size_cnt)

    # allocate larger demands first; only 3->4 is allowed as an exception
    for req in sorted(demand_size_cnt.keys(), reverse=True):
        need = int(demand_size_cnt[req])
        for _ in range(need):
            chosen = None
            if available[req] > 0:
                chosen = req
            elif req == 3 and available[4] > 0:
                chosen = 4
            else:
                return False
            available[chosen] -= 1
    return True

def verify_target_matches_milp(target: "ClusterState", milp_res: Dict[str, Any], verbose: bool = True):
    target_gpu_count = len(target.real_gpus())
    milp_gpu_count = len(extract_template_list_from_milp(milp_res))
    ok_gpu = (target_gpu_count == milp_gpu_count)

    target_cnt_exact = _collect_instance_multiset_from_target(target)
    milp_cnt = _collect_instance_multiset_from_milp(milp_res)
    ok_exact = (target_cnt_exact == milp_cnt)

    target_grouped = _group_multiset_by_workload_batch(target_cnt_exact)
    milp_grouped = _group_multiset_by_workload_batch(milp_cnt)

    ok_relaxed = True
    if set(target_grouped.keys()) != set(milp_grouped.keys()):
        ok_relaxed = False
    else:
        for key in milp_grouped:
            if not _relaxed_cover_ok_for_one_group(target_grouped[key], milp_grouped[key]):
                ok_relaxed = False
                break

    if verbose:
        print("=" * 80)
        print("[VERIFY] target vs milp")
        print("=" * 80)
        print(f"GPU count match              : {ok_gpu}  (target={target_gpu_count}, milp={milp_gpu_count})")
        print(f"Instance multiset exact      : {ok_exact}")
        print(f"Instance multiset relaxed    : {ok_relaxed}")
        if not ok_exact:
            print("\nTarget multiset:", target_cnt_exact)
            print("MILP multiset  :", milp_cnt)

    return ok_gpu and ok_relaxed


### Code Block 3: MILP extraction helpers


In [ ]:
# ============================================================
# [Transition Cell 3] MILP extraction helpers
# Role:
#   - convert MILP result into template list / instance demands / arrivals
#   - expand aggregated counts into per-instance demand objects
# ============================================================

def milp_expand_template_list(y_sol: Dict[int, int], templates=TEMPLATES) -> List[str]:
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        if int(cnt) <= 0:
            continue
        tpl_name = templates[int(t_idx)][0]
        out.extend([tpl_name] * int(cnt))
    return out

def extract_template_list_from_milp(milp_res: Dict[str, Any]) -> List[str]:
    if "chosen_templates" in milp_res and milp_res["chosen_templates"] is not None:
        return list(milp_res["chosen_templates"])
    if "y_sol" in milp_res and milp_res["y_sol"] is not None:
        return milp_expand_template_list(milp_res["y_sol"], TEMPLATES)
    raise ValueError("Cannot extract templates from milp_res")

def extract_instance_demands_from_milp(
    milp_res: Dict[str, Any],
    feasible_option_df_: Optional[Any] = None,
) -> List[Dict[str, Any]]:
    if feasible_option_df_ is None:
        feasible_option_df_ = feasible_option_df

    if "x_sol" not in milp_res or milp_res["x_sol"] is None:
        raise ValueError("milp_res does not contain x_sol")

    x_sol = milp_res["x_sol"]
    chosen = feasible_option_df_[feasible_option_df_["opt_idx"].isin(list(x_sol.keys()))].copy()

    agg = defaultdict(lambda: {"count": 0, "mu": None})
    for _, row in chosen.iterrows():
        opt_idx = int(row["opt_idx"])
        cnt = int(x_sol.get(opt_idx, 0))
        if cnt <= 0:
            continue
        key = (str(row["workload"]), str(row["profile"]), int(row["batch"]))
        agg[key]["count"] += cnt
        agg[key]["mu"] = float(row["mu"])

    out = []
    for (w, p, b), info in sorted(agg.items()):
        out.append({
            "workload": w,
            "profile": p,
            "batch": int(b),
            "count": int(info["count"]),
            "mu": float(info["mu"]),
        })
    return out

def _arrival_dict_from_milp(milp_res: Dict[str, Any]) -> Dict[str, float]:
    arr = {}
    if "arrival_rate" in milp_res and milp_res["arrival_rate"] is not None:
        vec = milp_res["arrival_rate"]
    else:
        vec = arrival_rate
    for i, w in enumerate(WORKLOAD_NAMES):
        arr[w] = float(vec[i])
    return arr

def _profile_need_from_instance_demands(instance_demands: List[Dict[str, Any]]) -> Dict[str, int]:
    c = Counter({p: 0 for p in PROFILE_ORDER})
    for d in instance_demands:
        c[d["profile"]] += int(d["count"])
    return dict(c)

def _expand_demands_with_ids(instance_demands: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    did = 0
    for d in instance_demands:
        for _ in range(int(d["count"])):
            out.append({
                "demand_id": did,
                "workload": d["workload"],
                "profile": d["profile"],
                "batch": int(d["batch"]),
                "mu": float(d["mu"]),
            })
            did += 1

    # harder-first ordering
    out.sort(key=lambda x: (-PROFILE_SIZE[x["profile"]], x["workload"], -x["batch"], x["demand_id"]))
    return out


### Code Block 4: Template-set search (abstract level)


In [ ]:
# ============================================================
# [Transition Cell 4] Template-set search (abstract level)
# Role:
#   - generate candidate abstract template multisets satisfying profile_need
#   - assign chosen templates to concrete GPU ids
# ============================================================

def _prev_real_template_list(prev_state: Optional["ClusterState"]) -> List[str]:
    if prev_state is None:
        return []
    return [g.template_str() for g in prev_state.real_gpus()]

def _template_usefulness_score(
    tpl: str,
    profile_need: Dict[str, int],
    prev_templates_counter: Counter,
    milp_templates_counter: Counter,
):
    cap = TEMPLATE_NAME_TO_K[tpl]
    cover = sum(min(cap[p], profile_need.get(p, 0)) * PROFILE_SIZE[p] for p in PROFILE_ORDER)
    return (
        1000 * (prev_templates_counter[tpl] > 0),
        500 * (milp_templates_counter[tpl] > 0),
        cover,
        -len(_template_to_parts(tpl)),
        tpl,
    )

def _dominates_need(cur_cap: Dict[str, int], profile_need: Dict[str, int]) -> bool:
    return all(cur_cap.get(p, 0) >= profile_need.get(p, 0) for p in PROFILE_ORDER)

def _enumerate_candidate_abstract_template_sets(
    gpu_count: int,
    profile_need: Dict[str, int],
    milp_template_ref: List[str],
    prev_state: Optional["ClusterState"],
    max_candidates: int = 64,
) -> List[List[str]]:
    all_templates = _template_name_list_from_globals()
    prev_templates = _prev_real_template_list(prev_state)
    prev_counter = Counter(prev_templates)
    milp_counter = Counter(milp_template_ref)

    pool = []
    seen = set()
    for tpl in prev_templates + milp_template_ref + all_templates:
        if tpl in TEMPLATE_NAME_TO_K and tpl not in seen:
            seen.add(tpl)
            pool.append(tpl)

    pool.sort(
        key=lambda tpl: _template_usefulness_score(
            tpl, profile_need, prev_counter, milp_counter
        ),
        reverse=True,
    )

    max_per_gpu = {p: 0 for p in PROFILE_ORDER}
    for tpl in pool:
        cap = TEMPLATE_NAME_TO_K[tpl]
        for p in PROFILE_ORDER:
            max_per_gpu[p] = max(max_per_gpu[p], cap[p])

    candidates = []

    def optimistic_reachable(cur_cap: Dict[str, int], remaining_gpus: int) -> bool:
        for p in PROFILE_ORDER:
            if cur_cap[p] + remaining_gpus * max_per_gpu[p] < profile_need[p]:
                return False
        return True

    def dfs(pos: int, chosen: List[str], cur_cap: Dict[str, int], start_idx: int):
        if len(candidates) >= max_candidates * 8:
            return
        if pos == gpu_count:
            if _dominates_need(cur_cap, profile_need):
                candidates.append(list(chosen))
            return
        if not optimistic_reachable(cur_cap, gpu_count - pos):
            return

        for idx in range(start_idx, len(pool)):
            tpl = pool[idx]
            cap = TEMPLATE_NAME_TO_K[tpl]
            nxt = dict(cur_cap)
            for p in PROFILE_ORDER:
                nxt[p] += cap[p]
            chosen.append(tpl)
            dfs(pos + 1, chosen, nxt, idx)
            chosen.pop()

    dfs(0, [], {p: 0 for p in PROFILE_ORDER}, 0)

    if not candidates:
        raise RuntimeError("No abstract template multiset can satisfy profile_need")

    def candidate_rank(cand):
        c = Counter(cand)
        over = sum(max(0, sum(TEMPLATE_NAME_TO_K[t][p] * c[t] for t in c) - profile_need[p]) for p in PROFILE_ORDER)
        return (
            sum(min(c[t], prev_counter[t]) for t in c),
            sum(min(c[t], milp_counter[t]) for t in c),
            -over,
            sorted(cand),
        )

    candidates.sort(key=candidate_rank, reverse=True)

    out = []
    seen2 = set()
    for cand in candidates:
        key = tuple(sorted(cand))
        if key in seen2:
            continue
        seen2.add(key)
        out.append(list(cand))
        if len(out) >= max_candidates:
            break
    return out

def _order_candidate_templates_for_gpu_ids(
    candidate_templates: List[str],
    gpu_count: int,
    prev_state: Optional["ClusterState"],
    milp_template_ref: List[str],
) -> List[str]:
    """
    Greedy O(G^2) assignment:
      - prefer matching old GPU template first
      - then matching MILP ref template on the same index
      - else use largest / more useful template
    """
    remain = list(candidate_templates)
    prev_templates = _prev_real_template_list(prev_state)
    ordered = []

    for gid in range(gpu_count):
        pick = None
        if gid < len(prev_templates) and prev_templates[gid] in remain:
            pick = prev_templates[gid]
        elif gid < len(milp_template_ref) and milp_template_ref[gid] in remain:
            pick = milp_template_ref[gid]
        else:
            remain.sort(key=lambda t: (len(_template_to_parts(t)), t))
            pick = remain[0]
        ordered.append(pick)
        remain.remove(pick)

    return ordered


### Code Block 5: Abstract -> physical expansion


In [ ]:
# ============================================================
# [Transition Cell 5] Abstract -> physical expansion
# Role:
#   - enumerate physical layouts for each abstract template
#   - score physical layouts against prev_state
#   - enumerate top physical-layout combinations for whole cluster
# ============================================================

def _get_prev_gpu_intervals(prev_state: Optional["ClusterState"], gpu_id: int):
    if prev_state is None:
        return []
    for g in prev_state.real_gpus():
        if g.gpu_id == gpu_id:
            return [(inst.start, inst.end, inst.profile) for inst in sorted(g.instances, key=lambda x: (x.start, x.end))]
    return []

def _layout_score_vs_prev(intervals: List[Tuple[int, int, str]], prev_intervals: List[Tuple[int, int, str]]):
    """
    4-tuple proxy:
      0) exact interval-profile matches
      1) same number of non-void intervals
      2) prefix matches by position
      3) total overlapping same-profile length
    """
    exact = 0
    prefix = 0
    overlap_same_profile = 0

    new_nv = [(s, e, p) for (s, e, p) in intervals if p != "void"]
    old_nv = [(s, e, p) for (s, e, p) in prev_intervals if p != "void"]

    for a in new_nv:
        for b in old_nv:
            if a == b:
                exact += 1

    for i in range(min(len(new_nv), len(old_nv))):
        if new_nv[i] == old_nv[i]:
            prefix += 1

    for s1, e1, p1 in new_nv:
        for s2, e2, p2 in old_nv:
            if p1 != p2:
                continue
            overlap = max(0, min(e1, e2) - max(s1, s2))
            overlap_same_profile += overlap

    return (
        int(exact),
        int(len(new_nv) == len(old_nv)),
        int(prefix),
        int(overlap_same_profile),
    )

def _get_physical_layout_candidates_for_gpu(
    abstract_template: str,
    gpu_id: int,
    prev_state: Optional["ClusterState"],
    topk: int = 4,
):
    prev_intervals = _get_prev_gpu_intervals(prev_state, gpu_id)
    cands = []
    for pstr, intervals in _all_unique_physical_realizations(abstract_template):
        score = _layout_score_vs_prev(intervals, prev_intervals)
        cands.append((pstr, intervals, score))
    cands.sort(key=lambda x: (x[2], x[0]), reverse=True)
    return cands[:topk]

def _enumerate_physical_layout_combinations(
    ordered_abstract_templates: List[str],
    prev_state: Optional["ClusterState"],
    milp_template_ref: List[str],
    max_combos: int = 32,
    per_gpu_topk: int = 4,
):
    per_gpu_candidates = []
    for gid, atpl in enumerate(ordered_abstract_templates):
        per_gpu_candidates.append(
            _get_physical_layout_candidates_for_gpu(
                abstract_template=atpl,
                gpu_id=gid,
                prev_state=prev_state,
                topk=per_gpu_topk,
            )
        )

    combos = []

    def dfs(gid, chosen_pstr, chosen_intervals, score_acc):
        if len(combos) >= max_combos * 8:
            return
        if gid == len(ordered_abstract_templates):
            combos.append((score_acc, list(chosen_pstr), list(chosen_intervals)))
            return
        for pstr, intervals, lscore in per_gpu_candidates[gid]:
            nxt = (
                score_acc[0] + lscore[0],
                score_acc[1] + lscore[1],
                score_acc[2] + lscore[2],
                score_acc[3] + lscore[3],
            )
            chosen_pstr.append(pstr)
            chosen_intervals.append(intervals)
            dfs(gid + 1, chosen_pstr, chosen_intervals, nxt)
            chosen_pstr.pop()
            chosen_intervals.pop()

    dfs(0, [], [], (0, 0, 0, 0))
    combos.sort(key=lambda x: (x[0], x[1]), reverse=True)

    out = []
    seen = set()
    for score_acc, pstrs, intervals in combos:
        key = tuple(pstrs)
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "physical_template_strs": list(pstrs),
            "intervals_list": list(intervals),
            "layout_score": tuple(score_acc),
        })
        if len(out) >= max_combos:
            break
    return out

def _make_slot_list_from_intervals_list(intervals_list: List[List[Tuple[int, int, str]]]) -> List[Dict[str, Any]]:
    slots = []
    sid = 0
    for gid, intervals in enumerate(intervals_list):
        for local_idx, (s, e, p) in enumerate(intervals):
            if p == "void":
                continue
            slots.append({
                "slot_id": sid,
                "gpu_id": gid,
                "slot_idx": local_idx,
                "start": s,
                "end": e,
                "profile": p,
            })
            sid += 1
    return slots


### Code Block 6: Common scoring / exact-preserve helpers


In [ ]:

# ============================================================
# [Transition Cell 6] Common scoring / preserve helpers
# Role:
#   - compute preserve / spread / collocation / mixed-GPU metrics
#   - build ClusterState from slot assignments
#   - provide unified score tuple used by both solvers
# Preserve definition:
#   same slot + same workload + same profile
#   (batch may differ)
# Extra:
#   rewrite void-like layouts after search:
#     3+3       -> 4+3
#     3+2+1     -> choose among 4+2+1 / 1+1+2+3 / 2+1+1+3
#     3+1+1+1   -> choose among 4+1+1+1 / 1+1+1+1+3
# ============================================================

def _has_prev(prev_state: Optional["ClusterState"]):
    return prev_state is not None and len(prev_state.real_gpus()) > 0

def _old_exact_slot_map(prev_state: Optional["ClusterState"]):
    mp = {}
    if prev_state is None:
        return mp
    for g in prev_state.real_gpus():
        for inst in g.instances:
            if inst.profile == "void":
                continue
            mp[(g.gpu_id, inst.start, inst.end, inst.profile)] = inst
    return mp

def _template_match_count(ordered_abstract_templates, ordered_physical_templates, prev_state):
    prev_templates = _prev_real_template_list(prev_state)
    if len(prev_templates) != len(ordered_abstract_templates):
        return 0
    c = 0
    for i in range(len(ordered_abstract_templates)):
        if i < len(prev_templates) and (
            prev_templates[i] == ordered_abstract_templates[i] or prev_templates[i] == ordered_physical_templates[i]
        ):
            c += 1
    return c

def _slot_preserve_match(slot: Dict[str, Any], demand: Dict[str, Any], prev_state: Optional["ClusterState"]) -> bool:
    old_map = _old_exact_slot_map(prev_state)
    old = old_map.get((slot["gpu_id"], slot["start"], slot["end"], slot["profile"]))
    if old is None:
        return False
    return old.workload == demand["workload"] and old.profile == demand["profile"]

def _inst_preserve_match(inst: "MigInstance", gpu_id: int, prev_state: Optional["ClusterState"]) -> bool:
    old_map = _old_exact_slot_map(prev_state)
    old = old_map.get((gpu_id, inst.start, inst.end, inst.profile))
    if old is None or inst.workload is None:
        return False
    return old.workload == inst.workload and old.profile == inst.profile

def _assignments_to_metrics(
    slots: List[Dict[str, Any]],
    assigned: Dict[int, Optional[Dict[str, Any]]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    layout_preserve_score: Tuple,
    prev_state: Optional["ClusterState"],
):
    exact_preserve = 0
    workload_gpus = defaultdict(set)
    per_gpu_workload_count = defaultdict(Counter)

    for sl in slots:
        info = assigned.get(sl["slot_id"], None)
        if info is None:
            continue
        d = info["demand"]
        workload_gpus[d["workload"]].add(sl["gpu_id"])
        per_gpu_workload_count[sl["gpu_id"]][d["workload"]] += 1

        if _slot_preserve_match(sl, d, prev_state):
            exact_preserve += 1

    spread = sum(len(v) for v in workload_gpus.values())

    collocate_pairs = 0
    mixed_gpu_count = 0
    for gid, cnt in per_gpu_workload_count.items():
        kinds = sum(1 for _, v in cnt.items() if v > 0)
        if kinds >= 2:
            mixed_gpu_count += 1
        for _, n in cnt.items():
            collocate_pairs += n * (n - 1) // 2

    return {
        "ordered_abstract_templates": list(ordered_abstract_templates),
        "ordered_physical_templates": list(ordered_physical_templates),
        "layout_preserve_score": tuple(layout_preserve_score),
        "exact_preserve": int(exact_preserve),
        "same_gpu_preserve": 0,
        "spread": int(spread),
        "collocate_pairs": int(collocate_pairs),
        "mixed_gpu_count": int(mixed_gpu_count),
        "template_match_count": int(_template_match_count(
            ordered_abstract_templates,
            ordered_physical_templates,
            prev_state,
        )),
    }

def _score_tuple(metrics: Dict[str, Any], prev_mode: bool):
    if prev_mode:
        return (
            metrics["exact_preserve"],
            -metrics["spread"],
            metrics["collocate_pairs"],
            -metrics["mixed_gpu_count"],
            metrics["layout_preserve_score"][0],
            metrics["layout_preserve_score"][1],
            metrics["template_match_count"],
        )
    else:
        return (
            -metrics["spread"],
            metrics["collocate_pairs"],
            -metrics["mixed_gpu_count"],
            metrics["layout_preserve_score"][0],
            metrics["template_match_count"],
        )

# ---------- layout rewrite helpers ----------

VOID_LIKE_REWRITE_CANDIDATES = {
    "3+3": [("4g", "3g")],
    "3+2+1": [("4g", "2g", "1g"), ("1g", "1g", "2g", "3g"), ("2g", "1g", "1g", "3g")],
    "3+1+1+1": [("4g", "1g", "1g", "1g"), ("1g", "1g", "1g", "1g", "3g")],
}

def _profiles_string_from_instances(insts: List["MigInstance"]) -> str:
    return "+".join(str(PROFILE_SIZE[inst.profile]) for inst in sorted(insts, key=lambda x: (x.start, x.end)) if inst.profile != "void")

def _candidate_priority_no_prev(current_t: str, profiles: Tuple[str, ...]) -> Tuple[int, ...]:
    s = _physical_profiles_to_string(profiles)
    # user preference:
    # 3+2+1 -> prefer 1+1+2+3
    # 3+1+1+1 -> prefer 1+1+1+1+3
    # for other cases use a stable fallback
    if current_t == "3+2+1":
        order = {"1+1+2+3": 0, "2+1+1+3": 1, "4+2+1": 2}
        return (order.get(s, 99),)
    if current_t == "3+1+1+1":
        order = {"1+1+1+1+3": 0, "4+1+1+1": 1}
        return (order.get(s, 99),)
    if current_t == "3+3":
        order = {"4+3": 0}
        return (order.get(s, 99),)
    return (99,)

def _build_intervals_from_profiles(profiles: Tuple[str, ...]) -> List[Tuple[int, int, str]]:
    return _physical_profiles_to_intervals(profiles)

def _assign_instances_to_candidate_layout(
    gpu_id: int,
    old_assigned_insts: List["MigInstance"],
    candidate_profiles: Tuple[str, ...],
    prev_state: Optional["ClusterState"],
    prefer_first_1g_idle: bool,
):
    """
    Remap currently assigned workloads on this GPU to a rewritten legal layout.
    Compatibility only for rewrite: a 3g workload may be placed on 4g.
    """
    intervals = _build_intervals_from_profiles(candidate_profiles)
    candidate_slots = []
    oneg_rank = 0
    for (s, e, p) in intervals:
        if p == "void":
            continue
        if p == "1g":
            oneg_rank += 1
            rank = oneg_rank
        else:
            rank = 0
        candidate_slots.append({
            "start": s, "end": e, "profile": p, "size": PROFILE_SIZE[p], "oneg_rank": rank
        })

    assigned_work = [inst for inst in sorted(old_assigned_insts, key=lambda x: (-PROFILE_SIZE[x.profile], x.start, x.end)) if inst.workload is not None]
    slot_used = [False] * len(candidate_slots)
    new_insts = []

    for inst in assigned_work:
        compat = []
        for idx, sl in enumerate(candidate_slots):
            if slot_used[idx]:
                continue
            if sl["size"] < PROFILE_SIZE[inst.profile]:
                continue

            preserve = 0
            if prev_state is not None:
                old = _old_exact_slot_map(prev_state).get((gpu_id, sl["start"], sl["end"], sl["profile"]))
                if old is not None and old.workload == inst.workload and old.profile == inst.profile:
                    preserve = 1

            exact_profile_fit = int(sl["profile"] == inst.profile)
            size_waste = sl["size"] - PROFILE_SIZE[inst.profile]

            # if no prev and we want first 1g idle, avoid occupying the first 1g when possible
            first_1g_penalty = 0
            if prev_state is None and prefer_first_1g_idle and inst.profile == "1g" and sl["profile"] == "1g":
                if sl["oneg_rank"] == 1:
                    first_1g_penalty = 1

            compat.append((
                preserve,
                exact_profile_fit,
                -size_waste,
                -first_1g_penalty,
                -sl["start"],
                -idx,
                idx,
            ))

        if not compat:
            return None

        compat.sort(reverse=True)
        chosen_idx = compat[0][-1]
        sl = candidate_slots[chosen_idx]
        slot_used[chosen_idx] = True

        new_insts.append(
            MigInstance(
                start=sl["start"],
                end=sl["end"],
                profile=sl["profile"],
                workload=inst.workload,
                batch=inst.batch,
                mu=inst.mu,
                preserved=False,  # recomputed below
            )
        )

    # add free slots
    for idx, sl in enumerate(candidate_slots):
        if not slot_used[idx]:
            new_insts.append(
                MigInstance(
                    start=sl["start"],
                    end=sl["end"],
                    profile=sl["profile"],
                    workload=None,
                    batch=None,
                    mu=0.0,
                    preserved=False,
                )
            )

    # add trailing/intermediate void if any (should already be edge-only here)
    new_insts = sorted(new_insts, key=lambda x: (x.start, x.end))
    cur = 0
    completed = []
    for inst in new_insts:
        if inst.start > cur:
            completed.append(MigInstance(start=cur, end=inst.start, profile="void", workload=None, batch=None, mu=0.0, preserved=False))
        completed.append(inst)
        cur = inst.end
    if cur < 7:
        completed.append(MigInstance(start=cur, end=7, profile="void", workload=None, batch=None, mu=0.0, preserved=False))

    # recompute preserved flags on rewritten intervals
    for inst in completed:
        if inst.workload is None or inst.profile == "void":
            inst.preserved = False
        else:
            inst.preserved = _inst_preserve_match(inst, gpu_id, prev_state)

    return completed

def _rewrite_void_like_layout_for_gpu(
    gpu_id: int,
    insts: List["MigInstance"],
    prev_state: Optional["ClusterState"],
):
    current_t = _profiles_string_from_instances(insts)
    if current_t not in VOID_LIKE_REWRITE_CANDIDATES:
        # still keep void on edges only
        return sorted(insts, key=lambda x: (x.start, x.end))

    candidates = VOID_LIKE_REWRITE_CANDIDATES[current_t]
    prev_intervals = _get_prev_gpu_intervals(prev_state, gpu_id)

    best = None
    best_key = None

    for cand_profiles in candidates:
        prefer_first_1g_idle = (prev_state is None and current_t in {"3+2+1", "3+1+1+1"})
        rewritten = _assign_instances_to_candidate_layout(
            gpu_id=gpu_id,
            old_assigned_insts=insts,
            candidate_profiles=cand_profiles,
            prev_state=prev_state,
            prefer_first_1g_idle=prefer_first_1g_idle,
        )
        if rewritten is None:
            continue

        preserve_cnt = sum(
            1 for inst in rewritten
            if inst.workload is not None and inst.profile != "void" and _inst_preserve_match(inst, gpu_id, prev_state)
        )

        intervals = [(inst.start, inst.end, inst.profile) for inst in rewritten]
        layout_score = _layout_score_vs_prev(intervals, prev_intervals)

        if prev_state is None:
            key = (
                0,  # all same preserve priority with no prev
                ) + tuple(-x for x in _candidate_priority_no_prev(current_t, cand_profiles))
        else:
            key = (
                preserve_cnt,
                layout_score[0],
                layout_score[1],
                layout_score[2],
                layout_score[3],
            )

        if best is None or key > best_key:
            best = rewritten
            best_key = key

    if best is None:
        return sorted(insts, key=lambda x: (x.start, x.end))
    return sorted(best, key=lambda x: (x.start, x.end))

def _assignments_to_target(
    slots: List[Dict[str, Any]],
    assigned: Dict[int, Optional[Dict[str, Any]]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    layout_preserve_score: Tuple,
    prev_state: Optional["ClusterState"],
):
    by_gpu = defaultdict(list)

    for sl in slots:
        info = assigned.get(sl["slot_id"], None)
        workload, batch, mu = None, None, 0.0
        preserved = False

        if info is not None:
            d = info["demand"]
            workload = d["workload"]
            batch = int(d["batch"])
            mu = float(d["mu"])
            preserved = _slot_preserve_match(sl, d, prev_state)

        by_gpu[sl["gpu_id"]].append(
            MigInstance(
                start=sl["start"], end=sl["end"], profile=sl["profile"],
                workload=workload, batch=batch, mu=mu, preserved=preserved
            )
        )

    gpus = []
    gpu_ids = sorted({sl["gpu_id"] for sl in slots})
    actual_physical_templates = []

    for gid in gpu_ids:
        insts = sorted(by_gpu[gid], key=lambda x: (x.start, x.end))

        # rewrite void-like layouts after search
        rewritten = _rewrite_void_like_layout_for_gpu(gid, insts, prev_state)
        gpus.append(GPUState(gpu_id=gid, source="real", instances=rewritten))
        actual_physical_templates.append(_profiles_string_from_instances(rewritten))

    target = ClusterState(gpus=gpus, metadata={})
    assert_valid_cluster_state(target)

    metrics = _assignments_to_metrics(
        slots=slots,
        assigned=assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=actual_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
    )
    # use the rewritten physical templates for reporting
    metrics["ordered_physical_templates"] = list(actual_physical_templates)
    return target, metrics

def _exact_preserve_preassign(
    demands: List[Dict[str, Any]],
    slots: List[Dict[str, Any]],
    prev_state: Optional["ClusterState"],
):
    """
    Hard-fix preserve using same slot + same workload + same profile.
    Batch does not matter.
    """
    old_map = _old_exact_slot_map(prev_state)

    demand_buckets = defaultdict(list)
    for d in demands:
        demand_buckets[(d["workload"], d["profile"])].append(d)

    # deterministic: use smaller batch first only as tie-break
    for k in demand_buckets:
        demand_buckets[k].sort(key=lambda d: (int(d["batch"]), int(d["demand_id"])))

    assigned = {sl["slot_id"]: None for sl in slots}
    preserved_ids = set()

    for sl in slots:
        old = old_map.get((sl["gpu_id"], sl["start"], sl["end"], sl["profile"]))
        if old is None or old.workload is None:
            continue
        key = (old.workload, old.profile)
        if len(demand_buckets[key]) == 0:
            continue
        d = demand_buckets[key].pop(0)
        assigned[sl["slot_id"]] = {"slot": sl, "demand": d}
        preserved_ids.add(d["demand_id"])

    residual_demands = [d for d in demands if d["demand_id"] not in preserved_ids]
    return assigned, residual_demands

def _group_demands_by_workload_profile(demands):
    buckets = defaultdict(list)
    for d in demands:
        buckets[(d["workload"], d["profile"])].append(d)
    return buckets


### Code Block 7: Preserve-first beam solver


In [ ]:

# ============================================================
# [Transition Cell 7] Preserve-first beam solver
# Role:
#   - preserve preprocessing
#   - beam search on residual problem
# ============================================================

@dataclass
class _BeamNode:
    assigned: Dict[int, Optional[Dict[str, Any]]]
    used_slots: set
    workload_gpus: Dict[str, set]
    per_gpu_workload_count: Dict[int, Counter]

    def clone(self):
        return _BeamNode(
            assigned=dict(self.assigned),
            used_slots=set(self.used_slots),
            workload_gpus=defaultdict(set, {k: set(v) for k, v in self.workload_gpus.items()}),
            per_gpu_workload_count=defaultdict(Counter, {g: Counter(c) for g, c in self.per_gpu_workload_count.items()}),
        )

def _beam_seed_from_preassign(slots, assigned):
    workload_gpus = defaultdict(set)
    per_gpu_workload_count = defaultdict(Counter)
    used = set()
    for sid, info in assigned.items():
        if info is None:
            continue
        sl = info["slot"]
        d = info["demand"]
        used.add(sid)
        workload_gpus[d["workload"]].add(sl["gpu_id"])
        per_gpu_workload_count[sl["gpu_id"]][d["workload"]] += 1
    return _BeamNode(
        assigned=dict(assigned),
        used_slots=used,
        workload_gpus=workload_gpus,
        per_gpu_workload_count=per_gpu_workload_count,
    )

def _order_residual_demands_for_search(demands, slots, prev_state):
    slots_by_profile = defaultdict(list)
    old_map = _old_exact_slot_map(prev_state)
    for sl in slots:
        slots_by_profile[sl["profile"]].append(sl)

    def preserve_candidates(d):
        cnt = 0
        for sl in slots_by_profile[d["profile"]]:
            old = old_map.get((sl["gpu_id"], sl["start"], sl["end"], sl["profile"]))
            if old is not None and old.workload == d["workload"] and old.profile == d["profile"]:
                cnt += 1
        return cnt

    ordered = list(demands)
    if _has_prev(prev_state):
        ordered.sort(key=lambda d: (-preserve_candidates(d), -PROFILE_SIZE[d["profile"]], d["workload"], int(d["batch"]), int(d["demand_id"])))
    else:
        ordered.sort(key=lambda d: (d["workload"], -PROFILE_SIZE[d["profile"]], int(d["batch"]), int(d["demand_id"])))
    return ordered

def _beam_slot_local_rank(demand, slot, node, prev_state):
    gpu_id = slot["gpu_id"]
    workload = demand["workload"]
    same_workload = node.per_gpu_workload_count[gpu_id][workload]
    new_touch = 1 if gpu_id not in node.workload_gpus[workload] else 0
    distinct_before = len(node.per_gpu_workload_count[gpu_id])
    mixed_penalty = 1 if (distinct_before >= 1 and same_workload == 0) else 0
    preserve_bonus = 1 if _slot_preserve_match(slot, demand, prev_state) else 0
    return (
        preserve_bonus,
        same_workload,
        -new_touch,
        -mixed_penalty,
        -slot["slot_idx"],
    )

def _beam_node_score(node, slots, ordered_abstract_templates, ordered_physical_templates, layout_preserve_score, prev_state):
    _, metrics = _assignments_to_target(
        slots=slots,
        assigned=node.assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=ordered_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
    )
    return _score_tuple(metrics, _has_prev(prev_state))

def _solve_target_with_preserve_first_beam(
    demands: List[Dict[str, Any]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    intervals_list: List[List[Tuple[int, int, str]]],
    prev_state: Optional["ClusterState"],
    beam_width: int = 32,
    slot_choice_width: int = 8,
    layout_preserve_score: Tuple = (0, 0, 0, 0),
):
    slots = _make_slot_list_from_intervals_list(intervals_list)
    assigned0, residual = _exact_preserve_preassign(demands, slots, prev_state)
    residual = _order_residual_demands_for_search(residual, slots, prev_state)
    beam = [_beam_seed_from_preassign(slots, assigned0)]

    slots_by_profile = defaultdict(list)
    for sl in slots:
        slots_by_profile[sl["profile"]].append(sl)

    for d in residual:
        next_beam = []
        cand_slots = slots_by_profile[d["profile"]]

        for node in beam:
            free_slots = [sl for sl in cand_slots if sl["slot_id"] not in node.used_slots]
            if not free_slots:
                continue

            ranked = sorted(
                free_slots,
                key=lambda sl: _beam_slot_local_rank(d, sl, node, prev_state),
                reverse=True,
            )[:slot_choice_width]

            for sl in ranked:
                child = node.clone()
                sid = sl["slot_id"]
                child.assigned[sid] = {"slot": sl, "demand": d}
                child.used_slots.add(sid)
                child.workload_gpus[d["workload"]].add(sl["gpu_id"])
                child.per_gpu_workload_count[sl["gpu_id"]][d["workload"]] += 1
                next_beam.append(child)

        if not next_beam:
            raise RuntimeError("Preserve-first beam failed: no feasible extension found.")

        next_beam.sort(
            key=lambda nd: _beam_node_score(
                node=nd,
                slots=slots,
                ordered_abstract_templates=ordered_abstract_templates,
                ordered_physical_templates=ordered_physical_templates,
                layout_preserve_score=layout_preserve_score,
                prev_state=prev_state,
            ),
            reverse=True,
        )
        beam = next_beam[:beam_width]

    best = max(
        beam,
        key=lambda nd: _beam_node_score(
            node=nd,
            slots=slots,
            ordered_abstract_templates=ordered_abstract_templates,
            ordered_physical_templates=ordered_physical_templates,
            layout_preserve_score=layout_preserve_score,
            prev_state=prev_state,
        ),
    )
    return _assignments_to_target(
        slots=slots,
        assigned=best.assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=ordered_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
    )


### Code Block 8: Greedy + move/swap local-repair solver


In [ ]:

# ============================================================
# [Transition Cell 8] Greedy + move/swap local-repair solver
# Role:
#   - preserve preprocessing
#   - workload-aware greedy assignment
#   - local repair with two neighborhoods:
#       (1) move, (2) swap
# Greedy policy:
#   - with prev_state: first preserve same slot+workload+profile,
#     then try to place remaining instances onto GPUs that already
#     carry the same workload
#   - without prev_state: directly pack same workload together
# ============================================================

def _order_residual_demands_for_greedy(demands, slots, prev_state):
    slots_by_profile = defaultdict(list)
    old_map = _old_exact_slot_map(prev_state)
    for sl in slots:
        slots_by_profile[sl["profile"]].append(sl)

    def preserve_candidates(d):
        cnt = 0
        for sl in slots_by_profile[d["profile"]]:
            old = old_map.get((sl["gpu_id"], sl["start"], sl["end"], sl["profile"]))
            if old is not None and old.workload == d["workload"] and old.profile == d["profile"]:
                cnt += 1
        return cnt

    ordered = list(demands)
    if _has_prev(prev_state):
        ordered.sort(key=lambda d: (-preserve_candidates(d), d["workload"], -PROFILE_SIZE[d["profile"]], int(d["batch"]), int(d["demand_id"])))
    else:
        ordered.sort(key=lambda d: (d["workload"], -PROFILE_SIZE[d["profile"]], int(d["batch"]), int(d["demand_id"])))
    return ordered

def _template_combo_bonus(demand, slot, assigned):
    """
    Mild bonus: if this GPU already holds the same workload on a different profile,
    encourage filling a more complete multi-profile bundle for that workload.
    """
    gpu_id = slot["gpu_id"]
    workload = demand["workload"]
    existing_profiles = set()
    for _, info in assigned.items():
        if info is None:
            continue
        sl = info["slot"]
        d = info["demand"]
        if sl["gpu_id"] == gpu_id and d["workload"] == workload:
            existing_profiles.add(d["profile"])

    if len(existing_profiles) == 0:
        return 0

    prof = demand["profile"]
    # reward adding a new profile type for same workload on same GPU
    return 1 if prof not in existing_profiles else 0

def _greedy_incremental_rank(demand, slot, assigned, prev_state):
    per_gpu_workload_count = defaultdict(Counter)
    workload_gpus = defaultdict(set)

    for _, info in assigned.items():
        if info is None:
            continue
        sl = info["slot"]
        d = info["demand"]
        per_gpu_workload_count[sl["gpu_id"]][d["workload"]] += 1
        workload_gpus[d["workload"]].add(sl["gpu_id"])

    gpu_id = slot["gpu_id"]
    workload = demand["workload"]

    preserve_bonus = 1 if _slot_preserve_match(slot, demand, prev_state) else 0
    same_workload = per_gpu_workload_count[gpu_id][workload]
    new_touch = 1 if gpu_id not in workload_gpus[workload] else 0
    distinct_before = len(per_gpu_workload_count[gpu_id])
    mixed_penalty = 1 if (distinct_before >= 1 and same_workload == 0) else 0
    combo_bonus = _template_combo_bonus(demand, slot, assigned)

    if _has_prev(prev_state):
        return (
            preserve_bonus,
            same_workload,
            combo_bonus,
            -new_touch,
            -mixed_penalty,
            -slot["slot_idx"],
        )
    else:
        return (
            same_workload,
            combo_bonus,
            -new_touch,
            -mixed_penalty,
            -slot["slot_idx"],
        )

def _solve_target_with_greedy_repair(
    demands: List[Dict[str, Any]],
    ordered_abstract_templates: List[str],
    ordered_physical_templates: List[str],
    intervals_list: List[List[Tuple[int, int, str]]],
    prev_state: Optional["ClusterState"],
    layout_preserve_score: Tuple = (0, 0, 0, 0),
    repair_rounds: int = 8,
):
    slots = _make_slot_list_from_intervals_list(intervals_list)
    assigned, residual = _exact_preserve_preassign(demands, slots, prev_state)
    residual = _order_residual_demands_for_greedy(residual, slots, prev_state)

    free_by_profile = defaultdict(list)
    for sl in slots:
        free_by_profile[sl["profile"]].append(sl)

    # ----- greedy fill -----
    for d in residual:
        cand = [sl for sl in free_by_profile[d["profile"]] if assigned[sl["slot_id"]] is None]
        if not cand:
            raise RuntimeError(f"Greedy failed: no free slot for ({d['workload']}, {d['profile']}, B{d['batch']})")
        cand.sort(key=lambda sl: _greedy_incremental_rank(d, sl, assigned, prev_state), reverse=True)
        best = cand[0]
        assigned[best["slot_id"]] = {"slot": best, "demand": d}

    # ----- local repair: move + swap -----
    prev_mode = _has_prev(prev_state)

    def is_preserved(sid, info):
        if info is None:
            return False
        return _slot_preserve_match(info["slot"], info["demand"], prev_state)

    _, cur_metrics = _assignments_to_target(
        slots=slots,
        assigned=assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=ordered_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
    )
    cur_score = _score_tuple(cur_metrics, prev_mode)

    slot_map = {sl["slot_id"]: sl for sl in slots}
    slot_ids = [sl["slot_id"] for sl in slots]

    for _ in range(repair_rounds):
        improved = False
        best_assign = dict(assigned)
        best_metrics = dict(cur_metrics)
        best_score = cur_score

        # (1) move neighborhood
        for src in slot_ids:
            src_info = assigned[src]
            if src_info is None or is_preserved(src, src_info):
                continue
            src_profile = src_info["slot"]["profile"]

            for dst in slot_ids:
                if assigned[dst] is not None:
                    continue
                if slot_map[dst]["profile"] != src_profile:
                    continue

                trial = dict(assigned)
                trial[dst] = {"slot": slot_map[dst], "demand": trial[src]["demand"]}
                trial[src] = None

                _, m = _assignments_to_target(
                    slots=slots,
                    assigned=trial,
                    ordered_abstract_templates=ordered_abstract_templates,
                    ordered_physical_templates=ordered_physical_templates,
                    layout_preserve_score=layout_preserve_score,
                    prev_state=prev_state,
                )
                s = _score_tuple(m, prev_mode)

                if s > best_score:
                    best_score = s
                    best_metrics = m
                    best_assign = trial
                    improved = True

        # (2) swap neighborhood
        for i in range(len(slot_ids)):
            sid1 = slot_ids[i]
            info1 = assigned[sid1]
            if info1 is None or is_preserved(sid1, info1):
                continue

            for j in range(i + 1, len(slot_ids)):
                sid2 = slot_ids[j]
                info2 = assigned[sid2]
                if info2 is None or is_preserved(sid2, info2):
                    continue
                if info1["slot"]["profile"] != info2["slot"]["profile"]:
                    continue

                trial = dict(assigned)
                slot1 = info1["slot"]
                slot2 = info2["slot"]

                trial[sid1] = {"slot": slot1, "demand": info2["demand"]}
                trial[sid2] = {"slot": slot2, "demand": info1["demand"]}

                _, m = _assignments_to_target(
                    slots=slots,
                    assigned=trial,
                    ordered_abstract_templates=ordered_abstract_templates,
                    ordered_physical_templates=ordered_physical_templates,
                    layout_preserve_score=layout_preserve_score,
                    prev_state=prev_state,
                )
                s = _score_tuple(m, prev_mode)

                if s > best_score:
                    best_score = s
                    best_metrics = m
                    best_assign = trial
                    improved = True

        if not improved:
            break

        assigned = best_assign
        cur_metrics = best_metrics
        cur_score = best_score

    return _assignments_to_target(
        slots=slots,
        assigned=assigned,
        ordered_abstract_templates=ordered_abstract_templates,
        ordered_physical_templates=ordered_physical_templates,
        layout_preserve_score=layout_preserve_score,
        prev_state=prev_state,
    )


### Code Block 9: Public API / method selector


In [ ]:
# ============================================================
# [Transition Cell 9] Public API / method selector
# Role:
#   - expose one unified entrypoint:
#       realize_target_state_search_from_milp(...)
#   - switch between preserve-first beam and greedy+repair
# ============================================================

TARGET_SEARCH_METHOD = "beam"   # "beam" or "greedy"

def set_target_search_method(method: str):
    global TARGET_SEARCH_METHOD
    method = str(method).lower().strip()
    if method not in {"beam", "greedy"}:
        raise ValueError("method must be 'beam' or 'greedy'")
    TARGET_SEARCH_METHOD = method
    print(f"[target-search] method = {TARGET_SEARCH_METHOD}")

def realize_target_state_search_from_milp(
    milp_res: Dict[str, Any],
    prev_state: Optional["ClusterState"] = None,
    max_candidate_templates: int = 64,
    beam_width: int = 32,
    slot_choice_width: int = 8,
    max_physical_layout_combos: int = 32,
    per_gpu_physical_topk: int = 4,
    verbose: bool = True,
) -> "ClusterState":
    t0 = time.time()

    milp_template_ref = extract_template_list_from_milp(milp_res)
    instance_demands = extract_instance_demands_from_milp(milp_res)
    arrivals = _arrival_dict_from_milp(milp_res)

    gpu_count = len(milp_template_ref)
    profile_need = _profile_need_from_instance_demands(instance_demands)
    demands = _expand_demands_with_ids(instance_demands)

    candidate_abstract_sets = _enumerate_candidate_abstract_template_sets(
        gpu_count=gpu_count,
        profile_need=profile_need,
        milp_template_ref=milp_template_ref,
        prev_state=prev_state,
        max_candidates=max_candidate_templates,
    )

    prev_mode = _has_prev(prev_state)
    best_target, best_metrics, best_score = None, None, None

    for cand_abs in candidate_abstract_sets:
        ordered_abs = _order_candidate_templates_for_gpu_ids(
            candidate_templates=cand_abs,
            gpu_count=gpu_count,
            prev_state=prev_state,
            milp_template_ref=milp_template_ref,
        )

        physical_layout_combos = _enumerate_physical_layout_combinations(
            ordered_abstract_templates=ordered_abs,
            prev_state=prev_state,
            milp_template_ref=milp_template_ref,
            max_combos=max_physical_layout_combos,
            per_gpu_topk=per_gpu_physical_topk,
        )

        for combo in physical_layout_combos:
            if TARGET_SEARCH_METHOD == "beam":
                target, metrics = _solve_target_with_preserve_first_beam(
                    demands=demands,
                    ordered_abstract_templates=ordered_abs,
                    ordered_physical_templates=combo["physical_template_strs"],
                    intervals_list=combo["intervals_list"],
                    prev_state=prev_state,
                    beam_width=beam_width,
                    slot_choice_width=slot_choice_width,
                    layout_preserve_score=combo["layout_score"],
                )
            else:
                target, metrics = _solve_target_with_greedy_repair(
                    demands=demands,
                    ordered_abstract_templates=ordered_abs,
                    ordered_physical_templates=combo["physical_template_strs"],
                    intervals_list=combo["intervals_list"],
                    prev_state=prev_state,
                    layout_preserve_score=combo["layout_score"],
                    repair_rounds=8,
                )

            score = _score_tuple(metrics, prev_mode)

            if best_score is None or score > best_score:
                best_score = score
                best_target = target
                best_metrics = metrics
                best_target.metadata["arrivals"] = dict(arrivals)

    if best_target is None:
        raise RuntimeError("Target-state search failed: no feasible candidate found")

    elapsed = time.time() - t0
    best_target.metadata["search_metrics"] = dict(best_metrics)
    best_target.metadata["search_metrics"]["score_tuple"] = best_score
    best_target.metadata["search_metrics"]["elapsed_time_sec"] = elapsed
    best_target.metadata["search_method"] = TARGET_SEARCH_METHOD

    if verbose:
        print("=" * 90)
        print(f"[TARGET-STATE SEARCH] BEST RESULT ({TARGET_SEARCH_METHOD})")
        print("=" * 90)
        print(f"GPU count fixed from MILP    : {gpu_count}")
        print(f"Profile need                : {profile_need}")
        print(f"MILP abstract template ref  : {milp_template_ref}")
        print(f"Chosen abstract templates   : {best_metrics['ordered_abstract_templates']}")
        print(f"Chosen physical templates   : {best_metrics['ordered_physical_templates']}")
        print(f"Layout preserve score       : {best_metrics['layout_preserve_score']}")
        print(f"Search score                : {best_score}")
        print(f"Search time (s)             : {elapsed:.4f}")
        print(f"exact_preserve              : {best_metrics['exact_preserve']}")
        print(f"same_gpu_preserve           : {best_metrics['same_gpu_preserve']}")
        print(f"spread                      : {best_metrics['spread']}")
        print(f"collocate_pairs             : {best_metrics['collocate_pairs']}")
        print(f"mixed_gpu_count             : {best_metrics['mixed_gpu_count']}")
        print(f"template_match_count        : {best_metrics['template_match_count']}")

    return best_target



### Code Block 10A: Target-search timing instrumentation

运行这个代码块后，可以给当前的 transition / target-search 流程加粗粒度计时统计。  
主要会统计：

- abstract candidate 枚举
- template ordering
- physical layout combination 枚举
- greedy / beam 主求解
- preserve preassign
- residual ordering
- greedy slot ranking
- assignments→target
- rewrite void-like layout
- repair 总时间

使用方式：

```python
enable_target_search_timing()
set_target_search_method("greedy")   # 或 "beam"
target0 = realize_target_state_search_from_milp(...)
print_target_search_timing_summary()
```

如果想关闭：

```python
disable_target_search_timing()
```


In [ ]:

# ============================================================
# Timing instrumentation for target-search
# ============================================================

from collections import defaultdict
import time
import functools

_TARGET_SEARCH_TIMING_ENABLED = False
_TARGET_SEARCH_TIMING = defaultdict(float)
_TARGET_SEARCH_TIMING_COUNT = defaultdict(int)
_TARGET_SEARCH_TIMING_ORIG = {}

def _timing_reset():
    global _TARGET_SEARCH_TIMING, _TARGET_SEARCH_TIMING_COUNT
    _TARGET_SEARCH_TIMING = defaultdict(float)
    _TARGET_SEARCH_TIMING_COUNT = defaultdict(int)

def _timing_add(name: str, dt: float):
    _TARGET_SEARCH_TIMING[name] += float(dt)
    _TARGET_SEARCH_TIMING_COUNT[name] += 1

def _make_timed_wrapper(fn_name, fn_obj):
    @functools.wraps(fn_obj)
    def wrapped(*args, **kwargs):
        t0 = time.time()
        out = fn_obj(*args, **kwargs)
        _timing_add(fn_name, time.time() - t0)
        return out
    return wrapped

def enable_target_search_timing():
    global _TARGET_SEARCH_TIMING_ENABLED, _TARGET_SEARCH_TIMING_ORIG

    if _TARGET_SEARCH_TIMING_ENABLED:
        _timing_reset()
        print("[timing] target-search timing already enabled; counters reset.")
        return

    _timing_reset()

    names_to_wrap = [
        "_enumerate_candidate_abstract_template_sets",
        "_order_candidate_templates_for_gpu_ids",
        "_enumerate_physical_layout_combinations",
        "_exact_preserve_preassign",
        "_order_residual_demands_for_greedy",
        "_greedy_incremental_rank",
        "_solve_target_with_greedy_repair",
        "_order_residual_demands_for_search",
        "_beam_slot_local_rank",
        "_solve_target_with_preserve_first_beam",
        "_assignments_to_target",
        "_rewrite_void_like_layout_for_gpu",
    ]

    _TARGET_SEARCH_TIMING_ORIG = {}
    g = globals()
    for name in names_to_wrap:
        if name in g and callable(g[name]):
            _TARGET_SEARCH_TIMING_ORIG[name] = g[name]
            g[name] = _make_timed_wrapper(name, g[name])

    # wrap public API too
    if "realize_target_state_search_from_milp" in g and callable(g["realize_target_state_search_from_milp"]):
        _TARGET_SEARCH_TIMING_ORIG["realize_target_state_search_from_milp"] = g["realize_target_state_search_from_milp"]

        @functools.wraps(g["realize_target_state_search_from_milp"])
        def _timed_realize_target_state_search_from_milp(*args, **kwargs):
            _timing_reset()
            t0 = time.time()
            out = _TARGET_SEARCH_TIMING_ORIG["realize_target_state_search_from_milp"](*args, **kwargs)
            total = time.time() - t0
            _timing_add("realize_target_state_search_from_milp", total)

            try:
                if hasattr(out, "metadata"):
                    out.metadata.setdefault("timing_breakdown", {})
                    out.metadata["timing_breakdown"] = dict(_TARGET_SEARCH_TIMING)
                    out.metadata["timing_breakdown_count"] = dict(_TARGET_SEARCH_TIMING_COUNT)
            except Exception:
                pass
            return out

        g["realize_target_state_search_from_milp"] = _timed_realize_target_state_search_from_milp

    _TARGET_SEARCH_TIMING_ENABLED = True
    print("[timing] target-search timing enabled.")

def disable_target_search_timing():
    global _TARGET_SEARCH_TIMING_ENABLED, _TARGET_SEARCH_TIMING_ORIG
    if not _TARGET_SEARCH_TIMING_ENABLED:
        print("[timing] target-search timing is not enabled.")
        return

    g = globals()
    for name, fn in _TARGET_SEARCH_TIMING_ORIG.items():
        g[name] = fn

    _TARGET_SEARCH_TIMING_ORIG = {}
    _TARGET_SEARCH_TIMING_ENABLED = False
    print("[timing] target-search timing disabled.")

def get_target_search_timing_breakdown():
    return dict(_TARGET_SEARCH_TIMING), dict(_TARGET_SEARCH_TIMING_COUNT)

def print_target_search_timing_summary(sort_by_time=True):
    timing, counts = get_target_search_timing_breakdown()
    if not timing:
        print("[timing] no timing data yet.")
        return

    items = list(timing.items())
    if sort_by_time:
        items.sort(key=lambda kv: kv[1], reverse=True)
    else:
        items.sort(key=lambda kv: kv[0])

    rows = []
    for name, sec in items:
        cnt = counts.get(name, 0)
        avg = sec / cnt if cnt else 0.0
        rows.append([name, sec, cnt, avg])

    print("=" * 90)
    print("Target-search timing summary")
    print("=" * 90)
    print(tabulate(
        rows,
        headers=["Function / Stage", "Total Time (s)", "Calls", "Avg / Call (s)"],
        tablefmt="github",
        floatfmt=".6f",
    ))


### Code Block 10: 实验代码
先运行 `set_target_search_method("beam")`，再跑 `target0/1/2`；
之后运行 `set_target_search_method("greedy")`，再跑同样的三段代码做对比。


In [ ]:

# ============================================================
# Helper 1: solve MILP under a new arrival-rate vector
# ============================================================

def _solve_milp_with_new_arrival_rate(
    new_arrival_rate,
    verbose=False,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    warm_start_res=None,
    apply_option_pruning=True,
):
    """
    Re-solve the same MILP with a new arrival-rate vector.
    Compatible with the current notebook's MILP setup.
    """
    arr = np.array(new_arrival_rate, dtype=float).copy()

    if arr.ndim != 1:
        raise ValueError("new_arrival_rate must be a 1-D array-like")
    if len(arr) != N_WORKLOADS:
        raise ValueError(f"new_arrival_rate length must be {N_WORKLOADS}, got {len(arr)}")

    res = solve_milp_gurobi_batch_unified(
        feasible_option_df=feasible_option_df,
        arrival_rate=arr,
        templates=TEMPLATES,
        template_K=TEMPLATE_K,
        profile_order=PROFILE_ORDER,
        n_workloads=N_WORKLOADS,
        time_limit_s=time_limit_s,
        mip_gap=mip_gap,
        threads=threads,
        verbose=verbose,
        apply_option_pruning=apply_option_pruning,
        warm_start_res=warm_start_res,
    )
    res["arrival_rate_used"] = arr.copy()
    return res


# ============================================================
# Helper 2: compare two arrival-rate vectors
# ============================================================

def print_arrival_rate_compare(old_arrival_rate, new_arrival_rate):
    old_arr = np.array(old_arrival_rate, dtype=float).copy()
    new_arr = np.array(new_arrival_rate, dtype=float).copy()

    if old_arr.ndim != 1 or new_arr.ndim != 1:
        raise ValueError("arrival-rate inputs must be 1-D array-like")
    if len(old_arr) != N_WORKLOADS or len(new_arr) != N_WORKLOADS:
        raise ValueError(f"arrival-rate length must be {N_WORKLOADS}")

    rows = []
    for i, w in enumerate(WORKLOAD_NAMES):
        old_v = float(old_arr[i])
        new_v = float(new_arr[i])
        ratio = (new_v / old_v) if old_v > 0 else np.nan
        delta = new_v - old_v
        pct = ((new_v - old_v) / old_v * 100.0) if old_v > 0 else np.nan
        rows.append([w, old_v, new_v, delta, ratio, pct])

    print("=" * 90)
    print("Arrival-rate comparison")
    print("=" * 90)
    print(tabulate(
        rows,
        headers=["Workload", "Old", "New", "Delta", "New/Old", "Delta %"],
        tablefmt="github",
        floatfmt=".4f"
    ))


In [ ]:
def _solve_milp_with_new_arrival_rate(
    new_arrival_rate,
    verbose=False,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    warm_start_res=None,
    apply_option_pruning=True,
):
    """
    Re-solve the MILP using a new arrival-rate vector.
    This compatibility wrapper forwards to solve_milp_gurobi_batch_unified.
    """
    arr = np.array(new_arrival_rate, dtype=float).copy()

    if arr.ndim != 1:
        raise ValueError("new_arrival_rate must be a 1-D array-like")
    if len(arr) != N_WORKLOADS:
        raise ValueError(f"new_arrival_rate length must be {N_WORKLOADS}, got {len(arr)}")

    return solve_milp_gurobi_batch_unified(
        feasible_option_df=feasible_option_df,
        arrival_rate=arr,
        templates=TEMPLATES,
        template_K=TEMPLATE_K,
        profile_order=PROFILE_ORDER,
        n_workloads=N_WORKLOADS,
        time_limit_s=time_limit_s,
        mip_gap=mip_gap,
        threads=threads,
        verbose=verbose,
        apply_option_pruning=apply_option_pruning,
        warm_start_res=warm_start_res,
    )

In [ ]:
# 选择算法："beam" 或 "greedy"
#set_target_search_method("beam")
enable_target_search_timing()
set_target_search_method("greedy")


[timing] target-search timing already enabled; counters reset.
[target-search] method = greedy


In [ ]:
target0 = realize_target_state_search_from_milp(
    milp_res=milp_res,
    prev_state=None,
    max_candidate_templates=64,
    beam_width=32,
    slot_choice_width=8,
    max_physical_layout_combos=32,
    per_gpu_physical_topk=4,
    verbose=True,
)

print_target_search_result(target0, title="[CHECK] TARGET STATE (no prev_state)")
verify_target_matches_milp(target0, milp_res)
print_target_search_timing_summary()
print("target0 templates:", [g.template_str() for g in target0.real_gpus()])

[TARGET-STATE SEARCH] BEST RESULT (greedy)
GPU count fixed from MILP    : 6
Profile need                : {'7g': 0, '4g': 3, '3g': 3, '2g': 1, '1g': 19}
MILP abstract template ref  : ['4+3', '4+1+1+1', '4+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '2+1+1+1+1+1']
Chosen abstract templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
Chosen physical templates   : ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
Layout preserve score       : (0, 0, 0, 0)
Search score                : (-8, 51, -1, 0, 0)
Search time (s)             : 0.8452
exact_preserve              : 0
same_gpu_preserve           : 0
spread                      : 8
collocate_pairs             : 51
mixed_gpu_count             : 1
template_match_count        : 0
[CHECK] TARGET STATE (no prev_state)
GPU 0: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 1: [0,4) 4g: llama / B1 / mu=0.7660 | [4,7) 3g: llama / B1 / mu=0.7410
GPU 2: [0,4) 4g: vit_base / B64 / m

In [ ]:
arrival_rate_up_manual = np.array(arrival_rate, dtype=float).copy()

name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}

arrival_rate_up_manual[name_to_idx["llama"]]    *= 1.15
arrival_rate_up_manual[name_to_idx["gpt2"]]     *= 1.12
arrival_rate_up_manual[name_to_idx["vgg16"]]    *= 1.08
arrival_rate_up_manual[name_to_idx["resnet50"]] *= 1.08
arrival_rate_up_manual[name_to_idx["vit_base"]] *= 3 #1.05

milp_res_up_manual = _solve_milp_with_new_arrival_rate(
    arrival_rate_up_manual,
    verbose=False,
    warm_start_res=milp_res,
)

print_arrival_rate_compare(arrival_rate, arrival_rate_up_manual)
print_milp_gurobi_batch_result(milp_res_up_manual)

target1 = realize_target_state_search_from_milp(
    milp_res=milp_res_up_manual,
    prev_state=target0,
    max_candidate_templates=64,
    beam_width=32,
    slot_choice_width=8,
    max_physical_layout_combos=32,
    per_gpu_physical_topk=4,
    verbose=True,
)

print_target_search_result(target1, title="[CHECK] TARGET STATE (prev_state = target0)")
verify_target_matches_milp(target1, milp_res_up_manual)
print_target_search_timing_summary()
print("target1 templates:", [g.template_str() for g in target1.real_gpus()])

Arrival-rate comparison
| Workload   |       Old |       New |     Delta |   New/Old |   Delta % |
|------------|-----------|-----------|-----------|-----------|-----------|
| llama      |    3.0000 |    3.4500 |    0.4500 |    1.1500 |   15.0000 |
| gpt2       |   20.0000 |   22.4000 |    2.4000 |    1.1200 |   12.0000 |
| vgg16      |  300.0000 |  324.0000 |   24.0000 |    1.0800 |    8.0000 |
| resnet50   |  300.0000 |  324.0000 |   24.0000 |    1.0800 |    8.0000 |
| vit_base   | 3000.0000 | 9000.0000 | 6000.0000 |    3.0000 |  200.0000 |
Method        : MILP-Gurobi(batch)
Elapsed (s)   : 0.0750
Feasible      : True
GPU count     : 9
Objective     : 9.0000
Templates     : ['4+3', '4+3', '2+2+3', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1']
K_total       : {'7g': 0, '4g': 2, '3g': 9, '2g': 2, '1g': 24}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)   |
|

In [ ]:
arrival_rate_t2_manual = np.array(arrival_rate_up_manual, dtype=float).copy()

name_to_idx = {name: i for i, name in enumerate(WORKLOAD_NAMES)}

# manually decrease load from target1 setting
arrival_rate_t2_manual[name_to_idx["llama"]]    *= 0.72
arrival_rate_t2_manual[name_to_idx["gpt2"]]     *= 0.78
arrival_rate_t2_manual[name_to_idx["vgg16"]]    *= 0.3
arrival_rate_t2_manual[name_to_idx["resnet50"]] *= 0.3
arrival_rate_t2_manual[name_to_idx["vit_base"]] *= 0.3

milp_res_t2_manual = _solve_milp_with_new_arrival_rate(
    arrival_rate_t2_manual,
    verbose=False,
    warm_start_res=milp_res_up_manual,
)

print_arrival_rate_compare(arrival_rate_up_manual, arrival_rate_t2_manual)
print_milp_gurobi_batch_result(milp_res_t2_manual)


target2 = realize_target_state_search_from_milp(
    milp_res=milp_res_t2_manual,
    prev_state=target1,   # important: target1 is now the old state
    max_candidate_templates=64,
    beam_width=32,
    slot_choice_width=8,
    max_physical_layout_combos=32,
    per_gpu_physical_topk=4,
    verbose=True,
)

print_target_search_result(target2, title="[CHECK] TARGET STATE (target2 from prev_state=target1)")
verify_target_matches_milp(target2, milp_res_t2_manual)
print_target_search_timing_summary()
print("target2 templates:", [g.template_str() for g in target2.real_gpus()])

Arrival-rate comparison
| Workload   |       Old |       New |      Delta |   New/Old |   Delta % |
|------------|-----------|-----------|------------|-----------|-----------|
| llama      |    3.4500 |    2.4840 |    -0.9660 |    0.7200 |  -28.0000 |
| gpt2       |   22.4000 |   17.4720 |    -4.9280 |    0.7800 |  -22.0000 |
| vgg16      |  324.0000 |   97.2000 |  -226.8000 |    0.3000 |  -70.0000 |
| resnet50   |  324.0000 |   97.2000 |  -226.8000 |    0.3000 |  -70.0000 |
| vit_base   | 9000.0000 | 2700.0000 | -6300.0000 |    0.3000 |  -70.0000 |
Method        : MILP-Gurobi(batch)
Elapsed (s)   : 0.0985
Feasible      : True
GPU count     : 6
Objective     : 6.0000
Templates     : ['4+3', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '3+1+1+1+1', '2+2+1+1+1']
K_total       : {'7g': 0, '4g': 1, '3g': 5, '2g': 2, '1g': 19}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)       |
|------------|-----

In [ ]:
print("target0 templates:", [g.template_str() for g in target0.real_gpus()])
print("target1 templates:", [g.template_str() for g in target1.real_gpus()])
print("target2 templates:", [g.template_str() for g in target2.real_gpus()])

target0 templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
target1 templates: ['4+3', '4+3', '4+3', '2+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+3', '4+3', '4+3']
target2 templates: ['4+3', '1+1+1+1+3', '4+3', '2+1+1+3', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
